# Integrated Experiment Runner — Main experiment: Recent Dataset, Final Project B Features (2-year walk-forward)

This notebook includes experiments using rolling two-year and expanding training windows across four targets: RL current PnL, RL long-action quality, RL long action, and perfect-hindsight positive long entry.

Additional target variables defined in this notebook are retained from earlier experiments and are not used in the final reported analysis.

**CI note.** The final dissertation results use 21-trading-day moving block bootstrap confidence intervals, applied within out-of-sample folds to account for temporal dependence. Alternative CI implementations retained later in this notebook are legacy/diagnostic versions only and are not used for the reported results; they are kept to illustrate sensitivity to alternative resampling assumptions and interval widths.

All training-window schemes use the same frozen 268-feature specification,
target definition and model set. The only experimental dimension varied here
is the length of the rolling training window.

## 0. Environment setup

Single install cell + single import/config cell (no duplicate installs or imports elsewhere).

In [1]:
# One-time installs (run once). Comment out if the environment already has these.
%pip install -q numpy pandas scikit-learn lightgbm scipy statsmodels matplotlib openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ---- imports (single source) ----
import os, json, math, re, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
                              HistGradientBoostingRegressor, HistGradientBoostingClassifier)
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             explained_variance_score, accuracy_score, balanced_accuracy_score,
                             f1_score, precision_score, recall_score, roc_auc_score,
                             average_precision_score, brier_score_loss, matthews_corrcoef,
                             cohen_kappa_score, log_loss, confusion_matrix)
warnings.filterwarnings('ignore')

# ---- reproducibility / parallelism ----
RANDOM_STATE = 42
N_JOBS = -1

# ---- canonical column names (shared by EDA and the runner) ----
INDEX_COL = 'IndexReference'
DATE_COL = 'attr__timestamp'
TICKER_COL = 'attr__ticker'
SIC2_COL = 'attr__sic2'
YEAR_COL = 'year'
SPLIT_COL = 'split'
TRUE_COL = 'y_true'
PRED_COL = 'y_pred'
SCORE_COL = 'prediction_score'
SIGNAL_SCORE_COL = 'signal_score'
DIRECTION_COL = 'direction'
CONFIDENCE_COL = 'confidence'
RL_TRAINABLE_COL = 'label__rl.trainable'

# ---- dataset selection (single knob) ----
UNIVERSE = 'universe_100'
TIME_HORIZON = 'recent'
NORMALISATION_STATUS = 'post_normalisation'  
# Change ONLY this constant to switch the whole notebook; file names follow
# f"{UNIVERSE}_{TIME_HORIZON}_{NORMALISATION_STATUS}_{split}.jsonl".

PROJECT_ROOT = Path.cwd()
DATASET_NAME = f"{UNIVERSE}_{TIME_HORIZON}_{NORMALISATION_STATUS}"
# Final Project B feature inventory (used by both the EDA and modelling sections).
PROJECT_B_FEATURE_FILE = PROJECT_ROOT / 'final_feature_B.txt'

# ---- output folders ----
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = PROJECT_ROOT / 'integrated_experiment_outputs_recent_final_feature_b'
TABLE_DIR = OUTPUT_DIR / 'tables'
PREDICTION_DIR = OUTPUT_DIR / 'predictions'
MODEL_DIR = OUTPUT_DIR / 'models'
FIGURE_DIR = OUTPUT_DIR / 'figures'
INVENTORY_DIR = OUTPUT_DIR / 'inventories'
for d in [OUTPUT_DIR, TABLE_DIR, PREDICTION_DIR, MODEL_DIR, FIGURE_DIR, INVENTORY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- EDA display constants ----
MISSING_THRESHOLD = 0.50
CORR_THRESHOLD = 0.95
TOP_MISSING_FEATURES_TO_PLOT = 30
TOP_VARIABLE_FEATURES_TO_PLOT = 20
TOP_CORRELATION_FEATURES = 30
HISTOGRAM_BINS = 50
FIGURE_DPI = 180
SELECTED_TARGETS = None

print('Output directory:', OUTPUT_DIR.resolve())

Output directory: C:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment_0806\integrated_experiment_outputs_recent_final_feature_b


## 1. Dataset configuration and loading

Defines the train/validation/test file paths and the final Project B feature file, then loads all three splits into `frames` and the pooled `all_data`.

*Applicable to:* all targets and all downstream sections.

In [3]:
# This section defines where the train, validation, and test files are located. 
# When switching between universe, time horizon, or pre/post-normalisation versions, only need to change the value of UNIVERSE/ TIME_HORIZON/ NORMALISATION_STATUS
DATASET_CONFIG = {
    "dataset_name": f"{UNIVERSE}_{TIME_HORIZON}_{NORMALISATION_STATUS}",
    "train_path": PROJECT_ROOT / f"{UNIVERSE}_{TIME_HORIZON}_{NORMALISATION_STATUS}_train.jsonl",
    "valid_path": PROJECT_ROOT / f"{UNIVERSE}_{TIME_HORIZON}_{NORMALISATION_STATUS}_validation.jsonl",
    "test_path":  PROJECT_ROOT / f"{UNIVERSE}_{TIME_HORIZON}_{NORMALISATION_STATUS}_test.jsonl",
    "project_b_feature_file": PROJECT_ROOT / "final_feature_B.txt",
    "universe": UNIVERSE,
    "time_horizon": TIME_HORIZON,
    "normalisation_setting": NORMALISATION_STATUS,
}
# Check whether all configured files exist
for key in ["train_path", "valid_path", "test_path", "project_b_feature_file"]:
    path = Path(DATASET_CONFIG[key])
    print(f"{key}: {path}")
    print("Exists:", path.exists())
    print("-" * 80)

train_path: c:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment_0806\universe_100_recent_post_normalisation_train.jsonl
Exists: True
--------------------------------------------------------------------------------
valid_path: c:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment_0806\universe_100_recent_post_normalisation_validation.jsonl
Exists: True
--------------------------------------------------------------------------------
test_path: c:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment_0806\universe_100_recent_post_normalisation_test.jsonl
Exists: True
--------------------------------------------------------------------------------
project_b_feature_file: c:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment_0806\final_feature_B.txt
Exists: True
--------------------------------------------------------------------------------


In [4]:
def load_flatten_jsonl(path):
    """
    Load and flatten an Adaptive Swarm model-ready JSONL file.

    Supported record structures:
    1. Official model-ready format:
       {"section": "data", "data": {"IndexReference": ..., "Attributes": {}, "Features": {}, "Labels": {}}}
    2. Already-flattened or earlier lower-case format:
       {"IndexReference": ..., "attributes": {}, "features": {}, "labels": {}}

    Output convention:
    - Attributes -> attr__*
    - Features   -> feature__*
    - Labels     -> label__*
    - IndexReference is preserved as the primary row join key for downstream prediction submission.
    """
    rows = []
    path = Path(path)

    def _open_text_file(p):
        if p.suffix.lower() == '.gz':
            import gzip
            return gzip.open(p, 'rt', encoding='utf-8')
        return p.open('r', encoding='utf-8')

    with _open_text_file(path) as f:
        for line in f:
            if not line.strip():
                continue

            raw_record = json.loads(line)

            # Skip header rows in official model-ready JSONL files.
            if raw_record.get('section') == 'header':
                continue

            # Official structure stores the useful row inside raw_record['data'].
            if raw_record.get('section') == 'data':
                record = raw_record.get('data', {}) or {}
            else:
                record = raw_record

            if not isinstance(record, dict):
                continue

            attrs = (
                record.get('Attributes')
                or record.get('attributes')
                or record.get('attrs')
                or {}
            )
            feats = record.get('Features') or record.get('features') or {}
            labs = record.get('Labels') or record.get('labels') or {}

            row = {INDEX_COL: record.get('IndexReference', raw_record.get('IndexReference'))}

            for k, v in attrs.items():
                row[k if str(k).startswith('attr__') else f'attr__{k}'] = v

            for k, v in feats.items():
                row[k if str(k).startswith('feature__') else f'feature__{k}'] = v

            for k, v in labs.items():
                row[k if str(k).startswith('label__') else f'label__{k}'] = v

            # Preserve simple scalar fields from the row for auditability.
            for k, v in record.items():
                if (
                    k not in ['Attributes', 'attributes', 'attrs', 'Features', 'features', 'Labels', 'labels']
                    and not isinstance(v, (dict, list))
                ):
                    row.setdefault(k, v)

            rows.append(row)

    df = pd.DataFrame(rows)
    print(f'Flattened JSONL shape: {df.shape}')
    return df


def load_jsonl_zip(path):
    """
    Load a .jsonl.zip file where the archive contains one JSONL file.
    This matches the project documentation's recommended model-ready data format.
    """
    import zipfile
    path = Path(path)
    rows = []

    with zipfile.ZipFile(path, 'r') as zf:
        jsonl_names = [name for name in zf.namelist() if name.lower().endswith('.jsonl')]
        if not jsonl_names:
            raise ValueError(f'No .jsonl file found inside zip archive: {path}')

        with zf.open(jsonl_names[0], 'r') as f:
            for raw_line in f:
                line = raw_line.decode('utf-8')
                if not line.strip():
                    continue
                raw_record = json.loads(line)
                if raw_record.get('section') == 'header':
                    continue
                record = raw_record.get('data', {}) if raw_record.get('section') == 'data' else raw_record
                if not isinstance(record, dict):
                    continue

                attrs = record.get('Attributes') or record.get('attributes') or record.get('attrs') or {}
                feats = record.get('Features') or record.get('features') or {}
                labs = record.get('Labels') or record.get('labels') or {}

                row = {INDEX_COL: record.get('IndexReference', raw_record.get('IndexReference'))}
                for k, v in attrs.items():
                    row[k if str(k).startswith('attr__') else f'attr__{k}'] = v
                for k, v in feats.items():
                    row[k if str(k).startswith('feature__') else f'feature__{k}'] = v
                for k, v in labs.items():
                    row[k if str(k).startswith('label__') else f'label__{k}'] = v
                for k, v in record.items():
                    if (
                        k not in ['Attributes', 'attributes', 'attrs', 'Features', 'features', 'Labels', 'labels']
                        and not isinstance(v, (dict, list))
                    ):
                        row.setdefault(k, v)
                rows.append(row)

    df = pd.DataFrame(rows)
    print(f'Flattened zipped JSONL shape: {df.shape}')
    return df


def load_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')

    name = path.name.lower()
    suffix = path.suffix.lower()

    if suffix == '.parquet':
        return pd.read_parquet(path)
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix == '.jsonl' or name.endswith('.jsonl.gz'):
        return load_flatten_jsonl(path)
    if name.endswith('.jsonl.zip'):
        return load_jsonl_zip(path)
    if suffix == '.json':
        try:
            return pd.read_json(path, lines=True)
        except ValueError:
            return pd.read_json(path)

    raise ValueError(f'Unsupported file format: {path}')


def standardise_frame(df, split_name):
    df = df.copy()
    df[SPLIT_COL] = split_name

    if INDEX_COL not in df.columns:
        # Fallback only for older flattened datasets without IndexReference.
        # Official simulator submission requires the source IndexReference, so this should be audited.
        df[INDEX_COL] = np.arange(len(df))
        print(f'Warning: {INDEX_COL} missing in {split_name}; generated sequential fallback index.')

    if DATE_COL in df.columns:
        df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
        df[YEAR_COL] = df[DATE_COL].dt.year

    for col in [TICKER_COL, SIC2_COL]:
        if col not in df.columns:
            df[col] = np.nan

    return df


def load_dataset_from_config(config):
    frames = {}
    for split_name in ['train', 'valid', 'test']:
        df = load_table(config[f'{split_name}_path'])
        frames[split_name] = standardise_frame(df, split_name)
        print(f'{split_name}: {frames[split_name].shape}')
    all_data = pd.concat(frames.values(), ignore_index=True, sort=False)
    return frames, all_data

frames, all_data = load_dataset_from_config(DATASET_CONFIG)

Flattened JSONL shape: (98419, 747)
train: (98419, 749)
Flattened JSONL shape: (24469, 747)
valid: (24469, 749)
Flattened JSONL shape: (32142, 747)
test: (32142, 749)


### Export manager

Single table registry (`register_table` / `EXPORTED_TABLES`) used by every section; one Excel workbook is written at the end.

In [5]:
if 'EXPORTED_TABLES' not in globals():
    EXPORTED_TABLES = {}


def safe_table_name(name, max_len=120):
    name = re.sub(r'[^A-Za-z0-9_\-]+', '_', str(name))
    name = re.sub(r'_+', '_', name).strip('_')
    return name[:max_len] or 'table'


def safe_sheet_name(name):
    name = re.sub(r'[\[\]\:\*\?\/\\]', '_', str(name))[:31]
    return name or 'Sheet'


def safe_file_name(name, max_len=120):
    name = str(name)
    for ch in ['\\', '/', ':', '*', '?', '"', '<', '>', '|']:
        name = name.replace(ch, '_')
    return name[:max_len] or 'file'


def make_excel_safe(df):
    """
    Convert DataFrame values into Excel-safe formats.
    The main fix is removing timezone information from datetime columns before Excel export.
    """
    df = df.copy()

    for col in df.columns:
        if pd.api.types.is_datetime64tz_dtype(df[col]):
            df[col] = df[col].dt.tz_convert(None)
        elif df[col].dtype == 'object':
            df[col] = df[col].apply(
                lambda x: x.tz_convert(None)
                if isinstance(x, pd.Timestamp) and x.tzinfo is not None
                else x
            )

    return df


def register_table(name, df, export_immediately=False):
    if df is None:
        return None
    if not isinstance(df, pd.DataFrame):
        df = pd.DataFrame(df)

    key = safe_table_name(name)
    EXPORTED_TABLES[key] = df.copy()

    if export_immediately:
        TABLE_DIR.mkdir(parents=True, exist_ok=True)
        path = TABLE_DIR / f'{key}.xlsx'
        make_excel_safe(df).to_excel(path, index=False)
        print('Exported:', path)

    return df


def export_registered_tables(workbook_name=None, export_individual_files=True, max_sheet_rows=1_000_000):
    if workbook_name is None:
        workbook_name = f'integrated_experiment_tables_{RUN_TIMESTAMP}.xlsx'

    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    workbook_path = TABLE_DIR / workbook_name

    used = set()
    n_written = 0
    skipped_rows = []

    with pd.ExcelWriter(workbook_path, engine='openpyxl') as writer:
        for name, df in EXPORTED_TABLES.items():
            if df is None:
                skipped_rows.append({'table_name': name, 'reason': 'df_is_none'})
                continue
            if not isinstance(df, pd.DataFrame):
                skipped_rows.append({'table_name': name, 'reason': f'not_dataframe_{type(df)}'})
                continue
            if len(df) == 0:
                skipped_rows.append({'table_name': name, 'reason': 'empty_dataframe'})
                continue

            sheet = safe_sheet_name(name)
            base = sheet
            i = 1
            while sheet in used:
                suffix = f'_{i}'
                sheet = safe_sheet_name(base[:31 - len(suffix)] + suffix)
                i += 1
            used.add(sheet)

            excel_df = make_excel_safe(df)
            excel_df.head(max_sheet_rows).to_excel(writer, sheet_name=sheet, index=False)
            n_written += 1

        # Excel workbooks must contain at least one visible sheet.
        if n_written == 0:
            readme = pd.DataFrame([
                {
                    'message': 'No non-empty registered tables were available for export.',
                    'possible_reason_1': 'EXPORTED_TABLES is empty.',
                    'possible_reason_2': 'The workflow cells that call register_table() have not been run.',
                    'possible_reason_3': 'All registered tables were empty.',
                    'next_step': 'Run EDA/model/diagnostic cells, then export again.',
                }
            ])
            readme.to_excel(writer, sheet_name='README', index=False)

    if export_individual_files:
        for name, df in EXPORTED_TABLES.items():
            if df is None or not isinstance(df, pd.DataFrame) or len(df) == 0:
                continue
            excel_df = make_excel_safe(df)
            excel_df.head(max_sheet_rows).to_excel(TABLE_DIR / f'{safe_file_name(name)}.xlsx', index=False)

    print('Integrated workbook exported to:', workbook_path.resolve())
    print('Number of registered tables:', len(EXPORTED_TABLES))
    print('Number of non-empty tables written:', n_written)

    if skipped_rows:
        display(pd.DataFrame(skipped_rows))

    return workbook_path

## 2. Label construction and target registry

Builds the `target__*` columns and the `TARGET_CONFIGS` registry (task type per target).

*Applicable labels:* perfect-hindsight (`pi_hindsight_entry_*`), RL long action/quality/reward, and `rl_long_current_pnl`.

In [6]:
def first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

########
def make_pi_hindsight_entry_long_6bins(series): 
    s = pd.to_numeric(series, errors='coerce')
    out = pd.Series(np.nan, index=s.index)
    out[s == 0] = 0
    out[(s > 0) & (s < 0.1)] = 1
    out[(s >= 0.1) & (s < 0.2)] = 2
    out[(s >= 0.2) & (s < 0.4)] = 3
    out[(s >= 0.4) & (s < 0.7)] = 4
    out[s >= 0.7] = 5
    return out.astype('Int64')


def add_derived_targets(df):
    df = df.copy()

    pi_col = first_existing_column(df, ['label__pi_hindsight_entry_long', 'label__pi_long_entry', 'pi_hindsight_entry_long'])
    if pi_col is not None:
        df['target__pi_hindsight_entry_long'] = pd.to_numeric(df[pi_col], errors='coerce')
        df['target__pi_hindsight_entry_positive'] = (df['target__pi_hindsight_entry_long'] > 0).astype('Int64')
        df['target__pi_hindsight_entry_original'] = (df['target__pi_hindsight_entry_long'] >= 0.4).astype('Int64')
        df['target__pi_hindsight_entry_6bins'] = make_pi_hindsight_entry_long_6bins(df['target__pi_hindsight_entry_long'])

    col = first_existing_column(df, ['label__rl.expert_action', 'label__rl_expert_action'])
    if col is not None:
        df['target__rl_expert_action'] = df[col]

    col = first_existing_column(df, ['label__rl.long_is_best', 'label__rl_long_is_best'])
    if col is not None:
        df['target__rl_long_is_best'] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    col = first_existing_column(df, ['label__rl.long.action_quality', 'label__rl_long_action_quality'])
    if col is not None:
        df['target__rl_long_action_quality'] = pd.to_numeric(df[col], errors='coerce')
    col = first_existing_column(df, ['label__rl.long.action_label'])
    if col is not None:
        df['target__rl_long_action_binary'] = pd.to_numeric(df[col], errors='coerce')        

    col = first_existing_column(df, ['label__rl.reward.long', 'label__rl_reward_long', 'label__rl.long.reward'])
    if col is not None:
        df['target__rl_long_reward'] = pd.to_numeric(df[col], errors='coerce')

    col = first_existing_column(df, ['label__rl.long.current_pnl', 'label__rl_long_current_pnl', 'label__rl.long_current_pnl', 'label__current_pnl'])
    if col is not None:
        df['target__rl_long_current_pnl'] = pd.to_numeric(df[col], errors='coerce')
        # Signed-log tames the heavy tail while preserving order & sign: y* = sign(y)*log(1+|y|)
        _pnl = df['target__rl_long_current_pnl']
        df['target__rl_long_current_pnl_signedlog'] = np.sign(_pnl) * np.log1p(_pnl.abs())

    reward_long = first_existing_column(df, ['label__rl.reward.long', 'target__rl_long_reward'])
    reward_short = first_existing_column(df, ['label__rl.reward.short', 'label__rl_reward_short'])
    reward_no_trade = first_existing_column(df, ['label__rl.reward.no_trade', 'label__rl_reward_no_trade', 'label__rl.reward.hold'])

    if reward_long and reward_short and reward_no_trade:
        r_long = pd.to_numeric(df[reward_long], errors='coerce')
        r_short = pd.to_numeric(df[reward_short], errors='coerce')
        r_no_trade = pd.to_numeric(df[reward_no_trade], errors='coerce')

        if 'target__rl_long_is_best' not in df.columns:
            df['target__rl_long_is_best'] = ((r_long > r_short) & (r_long > r_no_trade)).astype('Int64')

        best_non_long_reward = pd.concat([r_short, r_no_trade], axis=1).max(axis=1)
        df['target__rl_long_reward_margin'] = r_long - best_non_long_reward
        df['target__rl_long_high_confidence'] = (df['target__rl_long_reward_margin'] >= 0.05).astype('Int64')

    return df


TARGET_CONFIGS = {
    'pi_hindsight_entry_long': {'column': 'target__pi_hindsight_entry_long', 'task': 'regression', 'direction': 'higher_is_better', 'description': 'Continuous perfect-hindsight long-entry score.'},
    'pi_hindsight_entry_positive': {'column': 'target__pi_hindsight_entry_positive', 'task': 'binary', 'positive_label': 1, 'direction': 'higher_is_better', 'description': 'Binary target: pi score > 0.'},
    'pi_hindsight_entry_original': {'column': 'target__pi_hindsight_entry_original', 'task': 'binary', 'positive_label': 1, 'direction': 'higher_is_better', 'description': 'Binary target: pi score >= 0.4.'},
    'pi_hindsight_entry_6bins': {'column': 'target__pi_hindsight_entry_6bins', 'task': 'multiclass', 'direction': 'higher_is_better', 'description': 'Six-bin ordinal perfect-hindsight target.'},
    'rl_expert_action': {'column': 'target__rl_expert_action', 'task': 'multiclass', 'direction': 'action', 'description': 'RL evaluator expert action classification.'},
    'rl_long_action_binary': {'column': 'target__rl_long_action_binary', 'task': 'binary', 'positive_label': 1, 'description': 'RL evaluator long action classification.'},
    'rl_long_is_best': {'column': 'target__rl_long_is_best', 'task': 'binary', 'positive_label': 1, 'direction': 'higher_is_better', 'description': 'Binary long-side target: long is best action.'},
    'rl_long_high_confidence': {'column': 'target__rl_long_high_confidence', 'task': 'binary', 'positive_label': 1, 'direction': 'higher_is_better', 'description': 'Binary target: long reward exceeds the best non-long alternative by a margin.'},
    'rl_long_action_quality': {'column': 'target__rl_long_action_quality', 'task': 'regression', 'direction': 'higher_is_better', 'description': 'Continuous long action quality.'},
    'rl_long_reward': {'column': 'target__rl_long_reward', 'task': 'regression', 'direction': 'higher_is_better', 'description': 'Continuous long reward.'},
    'rl_long_reward_margin': {'column': 'target__rl_long_reward_margin', 'task': 'regression', 'direction': 'higher_is_better', 'description': 'Long reward minus the best non-long reward.'},
    'rl_long_current_pnl': {'column': 'target__rl_long_current_pnl', 'task': 'regression', 'direction': 'higher_is_better', 'description': 'Continuous long current PnL.'},
    'rl_long_current_pnl_signedlog': {'column': 'target__rl_long_current_pnl_signedlog', 'task': 'regression', 'direction': 'higher_is_better', 'description': 'Signed-log transform of current PnL (heavy-tail robustness target).'},
}


def apply_target_construction(frames):
    updated = {split: add_derived_targets(df) for split, df in frames.items()}
    all_data = pd.concat(updated.values(), ignore_index=True, sort=False)
    return updated, all_data

frames, all_data = apply_target_construction(frames)

In [7]:
def available_target_configs(df, target_configs=TARGET_CONFIGS):
    return {name: cfg for name, cfg in target_configs.items() if cfg['column'] in df.columns}


## 3. Feature inventory for modelling (full feature EDA have been moved to the EDA notebook)

In [8]:
# Minimal dependency for the modelling pipeline; descriptive feature EDA lives in the EDA notebook.
def normalise_feature_name(feature):
    feature = str(feature).strip()
    return feature if feature.startswith("feature__") else f"feature__{feature}"

if not PROJECT_B_FEATURE_FILE.exists():
    raise FileNotFoundError(f"Final Project B feature list not found: {PROJECT_B_FEATURE_FILE}")

final_feature_b_requested = list(dict.fromkeys(
    normalise_feature_name(line.strip())
    for line in PROJECT_B_FEATURE_FILE.read_text(encoding="utf-8").splitlines()
    if line.strip()
))
final_feature_b_available = [f for f in final_feature_b_requested if f in frames['train'].columns]
print("Requested features:", len(final_feature_b_requested),
      "| available in training:", len(final_feature_b_available))


Requested features: 270 | available in training: 270


## 4. Final feature set for modelling (`FINAL_FEATURE_B`)

The sanity-checked final Project B features (constants dropped) become the single modelling feature set used by **every** model and target.

In [9]:
# Drop constant / all-missing columns; this is the ONE feature set used everywhere.
FINAL_FEATURE_B = [c for c in final_feature_b_available
                   if c in frames['train'].columns and frames['train'][c].nunique(dropna=True) > 1]
print('final_feature_b requested:', len(final_feature_b_available),
      '| usable after dropping constants:', len(FINAL_FEATURE_B))

final_feature_b requested: 270 | usable after dropping constants: 268


In [10]:
EXCLUDE_COLUMNS = {SPLIT_COL, YEAR_COL, DATE_COL, TICKER_COL, SIC2_COL}


def read_feature_file(feature_file):
    if feature_file is None:
        return []
    path = Path(feature_file)
    if not path.exists():
        print(f'Feature file not found: {path}. Falling back to automatic numeric features.')
        return []
    raw = [line.strip() for line in path.open('r', encoding='utf-8') if line.strip() and not line.strip().startswith('#')]
    return [col if col.startswith('feature__') else f'feature__{col}' for col in raw]


def get_numeric_feature_candidates(df):
    numeric_cols = df.select_dtypes(include=[np.number, 'bool']).columns.tolist()
    out = []
    for col in numeric_cols:
        if col in EXCLUDE_COLUMNS:
            continue
        if col.startswith('label__') or col.startswith('target__') or col.startswith('attr__'):
            continue
        if col.startswith('feature__'):
            out.append(col)
    return sorted(set(out))


def select_features_by_keywords(all_features, include_keywords=None, exclude_keywords=None):
    include_keywords = include_keywords or []
    exclude_keywords = exclude_keywords or []
    out = []
    for col in all_features:
        c = col.lower()
        include_ok = True if not include_keywords else any(k.lower() in c for k in include_keywords)
        exclude_ok = not any(k.lower() in c for k in exclude_keywords)
        if include_ok and exclude_ok:
            out.append(col)
    return sorted(set(out))


def build_feature_sets(all_data, config=DATASET_CONFIG):
    auto_numeric = get_numeric_feature_candidates(all_data)
    project_b_raw = read_feature_file(config.get('project_b_feature_file'))
    project_b = [c for c in project_b_raw if c in all_data.columns]
    if len(project_b) == 0:
        project_b = auto_numeric.copy()

    feature_sets = {
        'final_feature_b': project_b,
        'combined_all_numeric_features': auto_numeric,
        'fundamentals_only': select_features_by_keywords(auto_numeric, ['fund', 'asset', 'liabil', 'equity', 'cash', 'debt', 'revenue', 'income', 'earn', 'profit', 'margin', 'eps', 'book', 'balance', 'report', 'quarter', 'ttm', 'filing']),
        'daily_valuation_only': select_features_by_keywords(auto_numeric, ['pe', 'pb', 'ps', 'ev', 'valuation', 'market_cap', 'price_to', 'yield', 'dividend', 'multiple']),
        'momentum_volatility_only': select_features_by_keywords(auto_numeric, ['return', 'ret', 'momentum', 'mom', 'vol', 'volatility', 'atr', 'rsi', 'macd', 'sma', 'ema', 'drawdown', 'trend', 'beta']),
        'macro_regime_only': select_features_by_keywords(auto_numeric, ['macro', 'regime', 'inflation', 'vix', 'yield_curve', 'rate', 'treasury', 'credit', 'index', 'sector', 'market']),
        'algorithmic_signals_only': select_features_by_keywords(auto_numeric, ['signal', 'alpha', 'score', 'rank', 'swarm', 'algo', 'model', 'entry', 'exit', 'confidence'], ['label', 'target']),
    }
    feature_sets = {k: v for k, v in feature_sets.items() if len(v) > 0}
    summary = pd.DataFrame([{'feature_set': k, 'n_features': len(v)} for k, v in feature_sets.items()])
    register_table('feature_set_summary', summary)
    return feature_sets

# build_feature_sets() is unused: the modelling pipeline uses FINAL_FEATURE_B (268) set below.

In [11]:
# Override: force the single 'final_feature_b' set for all experiments.
FEATURE_SET_CONFIGS = {'final_feature_b': FINAL_FEATURE_B}
register_table('feature_set_summary',
               pd.DataFrame([{'feature_set': 'final_feature_b', 'n_features': len(FINAL_FEATURE_B)}]))
FEATURE_SET_CONFIGS

{'final_feature_b': ['feature__accruals_to_assets',
  'feature__asset_turnover',
  'feature__avg_daily_turnover',
  'feature__b_asset_turnover_excess_over_p99',
  'feature__b_asset_turnover_industry_rank_score',
  'feature__b_asset_turnover_iqr',
  'feature__b_asset_turnover_is_above_p90',
  'feature__b_asset_turnover_is_below_p10',
  'feature__b_asset_turnover_shortfall_below_p1',
  'feature__b_assets_to_equity_excess_over_p99',
  'feature__b_assets_to_equity_iqr',
  'feature__b_assets_to_equity_is_above_p90',
  'feature__b_assets_to_equity_is_below_p10',
  'feature__b_assets_to_equity_shortfall_below_p1',
  'feature__b_cash_conversion_cycle_excess_over_p99',
  'feature__b_cash_conversion_cycle_iqr',
  'feature__b_cash_conversion_cycle_is_above_p90',
  'feature__b_cash_conversion_cycle_is_below_p10',
  'feature__b_cash_conversion_cycle_percentile_rank',
  'feature__b_cash_conversion_cycle_shortfall_below_p1',
  'feature__b_cfo_margin_excess_over_p99',
  'feature__b_cfo_margin_is_above

In [12]:
# ---- Feature-set leakage re-check (plan Section 1) ----
# Modelling features must not contain labels/targets or any forward/reward/
# simulator-outcome information. Hard-fail on label__/target__; flag keyword
# matches for manual review (some may be legitimate feature names).
_leak_hard = [c for c in FINAL_FEATURE_B if c.startswith(('label__', 'target__'))]
_leak_kw = [c for c in FINAL_FEATURE_B
            if re.search(r'(reward|pnl|future|forward|simulator|outcome|hindsight|_ahead|next_)', c.lower())]
print('HARD leakage (label__/target__ in features):', _leak_hard)
print('Keyword matches to review manually (may be legitimate):', _leak_kw)
register_table('feature_leakage_check', pd.DataFrame({
    'hard_leakage': pd.Series(_leak_hard, dtype=object),
    'keyword_review': pd.Series(_leak_kw, dtype=object)}))
assert not _leak_hard, 'Remove label/target columns from FINAL_FEATURE_B before modelling.'
print('Leakage check passed:', len(FINAL_FEATURE_B), 'modelling features.')

HARD leakage (label__/target__ in features): []
Keyword matches to review manually (may be legitimate): []
Leakage check passed: 268 modelling features.


## 5. Shared modelling utilities and models

This section defines the shared model factories used by the experiment runner.
The factories retain several additional candidate models from earlier
development stages for compatibility and reproducibility.

The final reported experiments are restricted through `EXPERIMENT_CONFIG`.
For continuous targets, the reported models are DummyMean, ElasticNet,
RandomForest and LightGBM. For binary targets, they are DummyMostFrequent,
Logistic Regression, RandomForest and LightGBM.

Ridge, Lasso, HistGradientBoosting, DummyStratified and multiclass model
definitions are retained as shared utilities but are not used in the final
reported experiments.

In [13]:
def make_numeric_preprocessor(scale=False):
    steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale:
        steps.append(('scaler', StandardScaler()))
    return Pipeline(steps)

def make_regression_models():
    models = {
        'DummyMean': DummyRegressor(strategy='mean'),
        'Ridge': make_pipeline(make_numeric_preprocessor(True), Ridge(alpha=1.0, random_state=RANDOM_STATE)),
        'Lasso': make_pipeline(make_numeric_preprocessor(True), Lasso(alpha=0.001, random_state=RANDOM_STATE, max_iter=5000)),
        'ElasticNet': make_pipeline(make_numeric_preprocessor(True), ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=RANDOM_STATE, max_iter=5000)),
        'RandomForest': make_pipeline(make_numeric_preprocessor(False), RandomForestRegressor(n_estimators=300, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=N_JOBS)),
        'HistGradientBoosting': make_pipeline(make_numeric_preprocessor(False), HistGradientBoostingRegressor(learning_rate=0.05, max_iter=300, l2_regularization=0.1, random_state=RANDOM_STATE)),
    }
    try:
        from lightgbm import LGBMRegressor
        models['LightGBM'] = make_pipeline(make_numeric_preprocessor(False), LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1))
    except Exception:
        pass
    return models


def make_binary_models():
    models = {
        'DummyMostFrequent': DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE),
        'DummyStratified': DummyClassifier(strategy='stratified', random_state=RANDOM_STATE),
        'Logistic': make_pipeline(make_numeric_preprocessor(True), LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)),
        'RandomForest': make_pipeline(make_numeric_preprocessor(False), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=N_JOBS)),
        'HistGradientBoosting': make_pipeline(make_numeric_preprocessor(False), HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, l2_regularization=0.1, random_state=RANDOM_STATE)),
    }
    try:
        from lightgbm import LGBMClassifier
        models['LightGBM'] = make_pipeline(make_numeric_preprocessor(False), LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1))
    except Exception:
        pass
    return models


def make_multiclass_models():
    models = {
        'DummyMostFrequent': DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE),
        'LogisticMultinomial': make_pipeline(make_numeric_preprocessor(True), LogisticRegression(class_weight='balanced', max_iter=3000, multi_class='auto', random_state=RANDOM_STATE)),
        'RandomForest': make_pipeline(make_numeric_preprocessor(False), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=N_JOBS)),
        'HistGradientBoosting': make_pipeline(make_numeric_preprocessor(False), HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, l2_regularization=0.1, random_state=RANDOM_STATE)),
    }
    try:
        from lightgbm import LGBMClassifier
        models['LightGBM'] = make_pipeline(make_numeric_preprocessor(False), LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1))
    except Exception:
        pass
    return models


def get_models_for_task(task):
    if task == 'regression':
        return make_regression_models()
    if task == 'binary':
        return make_binary_models()
    if task == 'multiclass':
        return make_multiclass_models()
    raise ValueError(f'Unknown task: {task}')


def prepare_xy(df, feature_cols, target_col, task):
    cols = [c for c in feature_cols if c in df.columns] + [target_col]
    data = df[cols].copy().dropna(subset=[target_col])
    X = data[[c for c in feature_cols if c in data.columns]]
    y = data[target_col]
    if task == 'regression':
        y = pd.to_numeric(y, errors='coerce')
        valid = y.notna()
        X = X.loc[valid]
        y = y.loc[valid]
    else:
        valid = y.notna()
        X = X.loc[valid]
        y = y.loc[valid]
    return X, y


def is_rl_target(target_name):
    return str(target_name).startswith('rl_')


def filter_rows_for_target(df, target_name):
    """
    Apply target-specific row filtering.
    RL-derived targets use label__rl.trainable == 1 when that flag exists.
    Non-RL targets are left unchanged.
    """
    df = df.copy()
    if is_rl_target(target_name) and RL_TRAINABLE_COL in df.columns:
        return df[df[RL_TRAINABLE_COL] == 1].copy()
    return df

def get_positive_proba(model, X):
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X)
        classes = getattr(model, 'classes_', None)
        if classes is None and hasattr(model, 'named_steps'):
            classes = getattr(list(model.named_steps.values())[-1], 'classes_', None)
        if classes is not None:
            classes = list(classes)
            if 1 in classes:
                return proba[:, classes.index(1)]
            if True in classes:
                return proba[:, classes.index(True)]
        return proba[:, -1]
    if hasattr(model, 'decision_function'):
        score = model.decision_function(X)
        return 1 / (1 + np.exp(-score))
    return None


def build_prediction_frame(df, y_true, y_pred, score, target_name, task, feature_set_name, model_name, split_name):
    """
    Build a row-level prediction DataFrame.
    IndexReference is preserved because it is the official join key for downstream simulation.
    """
    meta_cols = [INDEX_COL, DATE_COL, YEAR_COL, TICKER_COL, SIC2_COL]
    meta = df.loc[y_true.index, [c for c in meta_cols if c in df.columns]].copy()
    out = meta.copy()
    out['target_name'] = target_name
    out['task'] = task
    out['feature_set'] = feature_set_name
    out['model'] = model_name
    out[SPLIT_COL] = split_name
    out[TRUE_COL] = np.asarray(y_true)
    out[PRED_COL] = np.asarray(y_pred)
    out[SCORE_COL] = np.asarray(score) if score is not None else np.asarray(y_pred)
    return out


## 6. Metric functions

Regression, binary and multiclass metric blocks.

*Applicable labels:* regression metrics for continuous targets; ROC-AUC / PR-AUC / balanced-accuracy / F1 / MCC / Brier for binary targets.

In [14]:

def calibration_table(y_true, y_score, n_bins=10):
    df = pd.DataFrame({TRUE_COL: pd.Series(y_true).astype(float), SCORE_COL: pd.Series(y_score).astype(float)}).dropna()
    if len(df) == 0:
        return pd.DataFrame()
    df[SCORE_COL] = df[SCORE_COL].clip(0, 1)
    df['prob_bin'] = pd.cut(df[SCORE_COL], bins=np.linspace(0, 1, n_bins + 1), include_lowest=True)
    out = df.groupby('prob_bin', observed=False).agg(
        n=(TRUE_COL, 'size'),
        mean_predicted_probability=(SCORE_COL, 'mean'),
        true_positive_rate=(TRUE_COL, 'mean'),
    ).reset_index()
    out['calibration_error'] = out['mean_predicted_probability'] - out['true_positive_rate']
    return out


def daily_cross_sectional_spearman(pred_df, min_daily_rows=5):
    rows = []
    if DATE_COL not in pred_df.columns:
        return pd.DataFrame()
    for date, g in pred_df.groupby(DATE_COL):
        if len(g) < min_daily_rows:
            continue
        rows.append({
            DATE_COL: date,
            'n': len(g),
            'daily_spearman': safe_spearman(g[TRUE_COL], g[SCORE_COL]),
            'mean_true': pd.to_numeric(g[TRUE_COL], errors='coerce').mean(),
            'mean_score': pd.to_numeric(g[SCORE_COL], errors='coerce').mean(),
        })
    return pd.DataFrame(rows)


def topk_diagnostics(pred_df, top_pct=0.10, min_daily_rows=10):
    rows = []
    if DATE_COL not in pred_df.columns:
        return pd.DataFrame()
    for date, g in pred_df.groupby(DATE_COL):
        g = g.dropna(subset=[TRUE_COL, SCORE_COL]).copy()
        if len(g) < min_daily_rows:
            continue
        k = max(1, int(math.ceil(len(g) * top_pct)))
        top = g.nlargest(k, SCORE_COL)
        bottom = g.nsmallest(k, SCORE_COL)
        y = pd.to_numeric(g[TRUE_COL], errors='coerce')
        top_y = pd.to_numeric(top[TRUE_COL], errors='coerce')
        bottom_y = pd.to_numeric(bottom[TRUE_COL], errors='coerce')
        row = {
            DATE_COL: date,
            'n': len(g),
            'k': k,
            'top_pct': top_pct,
            'overall_mean_true': y.mean(),
            'top_mean_true': top_y.mean(),
            'bottom_mean_true': bottom_y.mean(),
            'top_minus_bottom_spread': top_y.mean() - bottom_y.mean(),
            'overall_median_true': y.median(),
            'top_median_true': top_y.median(),
            'bottom_median_true': bottom_y.median(),
            'top_minus_bottom_spread_median': top_y.median() - bottom_y.median(),
            'top_minus_overall_mean': top_y.mean() - y.mean(),   # lift vs overall (works for continuous)
        }
        unique_values = pd.Series(g[TRUE_COL]).dropna().unique()
        if set(unique_values).issubset({0, 1, False, True}):
            base_rate = y.mean()
            precision_at_k = top_y.mean()
            row.update({
                'base_positive_rate': base_rate,
                'precision_at_k': precision_at_k,
                'bottom_positive_rate': bottom_y.mean(),
                'lift_at_k': precision_at_k / base_rate if base_rate and base_rate > 0 else np.nan,
            })
        rows.append(row)
    return pd.DataFrame(rows)


def within_ticker_ranking(pred_df, min_rows=20):
    if TICKER_COL not in pred_df.columns:
        return pd.DataFrame()
    rows = []
    for ticker, g in pred_df.groupby(TICKER_COL, dropna=False):
        if len(g) < min_rows:
            continue
        rows.append({
            'ticker': ticker,
            'n': len(g),
            'within_ticker_spearman': safe_spearman(g[TRUE_COL], g[SCORE_COL]),
            'mean_true': pd.to_numeric(g[TRUE_COL], errors='coerce').mean(),
            'mean_score': pd.to_numeric(g[SCORE_COL], errors='coerce').mean(),
        })
    return pd.DataFrame(rows)


def subgroup_prediction_metrics(pred_df, group_col, min_rows=30):
    if group_col not in pred_df.columns:
        return pd.DataFrame()
    rows = []
    for group_value, g in pred_df.groupby(group_col, dropna=False):
        if len(g) < min_rows:
            continue
        row = {
            'group_col': group_col,
            'group_value': group_value,
            'n': len(g),
            'mean_true': pd.to_numeric(g[TRUE_COL], errors='coerce').mean(),
            'mean_score': pd.to_numeric(g[SCORE_COL], errors='coerce').mean(),
            'spearman': safe_spearman(g[TRUE_COL], g[SCORE_COL]),
        }
        unique_values = pd.Series(g[TRUE_COL]).dropna().unique()
        try:
            if set(unique_values).issubset({0, 1, False, True}):
                row.update(binary_metrics(g[TRUE_COL], g[PRED_COL], g[SCORE_COL]))
            else:
                row.update(regression_metrics(g[TRUE_COL], g[SCORE_COL]))
        except Exception:
            pass
        rows.append(row)
    return pd.DataFrame(rows)

## 7. Dependence-robust inference layer (CIs + paired tests)

**Alternative inference note.** This section retains earlier inference utilities,
including alternative bootstrap and Newey-West implementations, for diagnostic
and sensitivity comparisons. These estimates are not used as the final
within-ticker confidence intervals reported in the dissertation. The final
within-ticker CIs are calculated in Section 12.3 using a fold-stratified
21-trading-day moving-block bootstrap.
Ranking and classification metrics on daily panel data are **not** independent across observations, so naive standard errors are too small. This section adds:

* **Ranking (regression targets):** pooled / daily-cross-sectional / within-ticker Spearman with
  * **Newey-West** CI for the daily-cross-sectional series (cross-ticker same-day correlation is absorbed by collapsing to one number per day; NW handles day-to-day autocorrelation), and
  * **moving-block bootstrap over dates** for pooled and within-ticker (blocks keep within-ticker autocorrelation; keeping all tickers of a sampled date together keeps cross-ticker correlation).
* **Classification (binary targets):** ROC-AUC / PR-AUC / balanced-accuracy / F1 with **block-bootstrap-over-dates** CIs, at a threshold **tuned on the validation split**.

*Applicable labels:* Spearman family → continuous targets only (`rl_long_current_pnl`, `pi_hindsight_entry_long`); classification block → binary targets (`rl_long_action_binary`, `pi_hindsight_entry_positive`).

In [ ]:
# ============================================================================
# Alternative inference utilities retained for diagnostic comparisons
# ============================================================================
# These functions are not used for the final reported within-ticker CIs.

# ============================================================================
# Dependence-robust inference: Spearman (block-bootstrap / Newey-West) +
# classification metrics with block-bootstrap CIs + paired model comparison.
# Uses only numpy / pandas / sklearn.metrics and the notebook column constants.
# ============================================================================

# ---- fast Spearman (average ranks; NaN on degenerate/binary, like the runner) ----
def _avg_rank(a):
    n = a.size
    order = np.argsort(a, kind='mergesort'); sa = a[order]
    ordinal = np.arange(1, n + 1, dtype=float)
    grp = np.cumsum(np.r_[True, sa[1:] != sa[:-1]]) - 1
    avg = np.bincount(grp, weights=ordinal) / np.bincount(grp)
    r = np.empty(n); r[order] = avg[grp]
    return r

def _rho(t, s, min_valid=3):
    m = np.isfinite(t) & np.isfinite(s)
    if m.sum() < min_valid:
        return np.nan
    t, s = t[m], s[m]
    if np.unique(t).size < 2 or np.unique(s).size < 2:
        return np.nan
    rt, rs = _avg_rank(t), _avg_rank(s)
    rt -= rt.mean(); rs -= rs.mean()
    d = np.sqrt((rt @ rt) * (rs @ rs))
    return float((rt @ rs) / d) if d > 0 else np.nan

def _grouped_rho(t, s, codes, min_rows):
    order = np.argsort(codes, kind='mergesort')
    c, ts, ss = codes[order], t[order], s[order]
    bnd = np.flatnonzero(np.r_[True, c[1:] != c[:-1], True])
    out = []
    for i in range(bnd.size - 1):
        a, b = bnd[i], bnd[i + 1]
        if b - a >= min_rows:
            r = _rho(ts[a:b], ss[a:b])
            if np.isfinite(r):
                out.append(r)
    return np.asarray(out, float)


class SpearmanInference:
    """Point estimates + dependence-robust CIs for pooled / daily-cross-sectional
    / within-ticker Spearman, plus paired model comparison."""
    def __init__(self, cols=(TICKER_COL, DATE_COL, SCORE_COL, TRUE_COL),
                 block=21, B=1000, agg='median', maxlags=None,
                 min_ticker_rows=20, min_daily_rows=5, alpha=0.05, seed=0):
        self.ticker, self.date, self.score, self.true = cols
        self.block, self.B, self.agg = block, B, agg
        self.maxlags, self.alpha, self.seed = maxlags, alpha, seed
        self.min_ticker_rows, self.min_daily_rows = min_ticker_rows, min_daily_rows

    def _agg(self, x):
        x = np.asarray(x, float); x = x[np.isfinite(x)]
        return np.nan if x.size == 0 else float(np.median(x) if self.agg == 'median' else np.mean(x))

    def _prep(self, d):
        d = d.dropna(subset=[self.date, self.score, self.true])
        s = pd.to_numeric(d[self.score], errors='coerce').to_numpy(float)
        t = pd.to_numeric(d[self.true], errors='coerce').to_numpy(float)
        tk = pd.factorize(d[self.ticker].to_numpy())[0]
        dc, du = pd.factorize(d[self.date].to_numpy(), sort=True)
        order = np.argsort(dc, kind='mergesort'); cc = dc[order]
        bnd = np.flatnonzero(np.r_[True, cc[1:] != cc[:-1], True])
        pos = [order[bnd[i]:bnd[i + 1]] for i in range(bnd.size - 1)]
        return dict(s=s, t=t, tk=tk, dc=dc, n=len(du), pos=pos)

    def _pooled(self, P, idx=None):
        return _rho(P['t'], P['s']) if idx is None else _rho(P['t'][idx], P['s'][idx])
    def _daily(self, P, idx=None):
        if idx is None:
            return _grouped_rho(P['t'], P['s'], P['dc'], self.min_daily_rows)
        return _grouped_rho(P['t'][idx], P['s'][idx], P['dc'][idx], self.min_daily_rows)
    def _within(self, P, idx=None):
        if idx is None:
            return self._agg(_grouped_rho(P['t'], P['s'], P['tk'], self.min_ticker_rows))
        return self._agg(_grouped_rho(P['t'][idx], P['s'][idx], P['tk'][idx], self.min_ticker_rows))

    def _nw(self, x):
        # Newey-West (Bartlett) long-run variance of the mean: S = g0 + 2*sum w_k*g_k
        x = np.asarray(x, float); x = x[np.isfinite(x)]; n = x.size
        if n < 3:
            return (np.nan, np.nan, np.nan, np.nan)
        L = self.maxlags if self.maxlags is not None else int(np.floor(4 * (n / 100.) ** (2 / 9)))
        L = max(1, min(L, n - 1)); e = x - x.mean(); S = (e @ e) / n
        for k in range(1, L + 1):
            S += 2. * (1. - k / (L + 1.)) * (e[k:] @ e[:-k]) / n
        se = float(np.sqrt(max(S / n, 0.))); m = float(x.mean()); z = 1.959963985
        return (m, se, m - z * se, m + z * se)

    def _sample(self, P, rng):
        L, n = self.block, P['n']
        starts = np.arange(0, max(1, n - L + 1))
        seq = np.concatenate([np.arange(c, min(c + L, n))
                              for c in rng.choice(starts, size=int(np.ceil(n / L)), replace=True)])
        return np.concatenate([P['pos'][c] for c in seq]), seq

    def _boot(self, P, stat):
        rng = np.random.default_rng(self.seed); out = np.empty(self.B)
        for b in range(self.B):
            idx, _ = self._sample(P, rng); out[b] = stat(P, idx)
        out = out[np.isfinite(out)]
        lo, hi = np.percentile(out, [100 * self.alpha / 2, 100 * (1 - self.alpha / 2)])
        return float(np.std(out, ddof=1)), float(lo), float(hi)

    def summary(self, d):
        P = self._prep(d); rows = []
        pt = self._pooled(P); se, lo, hi = self._boot(P, self._pooled)
        rows.append(('pooled_spearman', pt, se, lo, hi, f'block-bootstrap B={self.B}'))
        dr = self._daily(P); m, se, lo, hi = self._nw(dr)
        rows.append(('daily_cs_spearman', m, se, lo, hi, f'Newey-West, n_days={dr.size}'))
        wt = _grouped_rho(P['t'], P['s'], P['tk'], self.min_ticker_rows)
        pt = self._agg(wt); se, lo, hi = self._boot(P, self._within)
        rows.append((f'within_ticker_spearman_{self.agg}', pt, se, lo, hi, f'block-bootstrap, n_tickers={wt.size}'))
        rows.append(('within_ticker_positive_rate', float((wt > 0).mean()) if wt.size else np.nan,
                     np.nan, np.nan, np.nan, f'n_tickers={wt.size}'))
        return pd.DataFrame(rows, columns=['metric', 'point', 'se', 'ci_low', 'ci_high', 'method'])

    def compare(self, da, db, la='A', lb='B'):
        # paired difference (A-B) on identical resampled dates
        common = np.intersect1d(da[self.date].unique(), db[self.date].unique())
        a = da[da[self.date].isin(common)]; b = db[db[self.date].isin(common)]
        cats = pd.CategoricalDtype(np.sort(common), ordered=True)
        def prep(d):
            d = d.dropna(subset=[self.date, self.score, self.true])
            s = pd.to_numeric(d[self.score], errors='coerce').to_numpy(float)
            t = pd.to_numeric(d[self.true], errors='coerce').to_numpy(float)
            tk = pd.factorize(d[self.ticker].to_numpy())[0]
            dc = d[self.date].astype(cats).cat.codes.to_numpy(); n = len(cats.categories)
            order = np.argsort(dc, kind='mergesort'); cc = dc[order]
            bnd = np.flatnonzero(np.r_[True, cc[1:] != cc[:-1], True])
            pmap = {int(cc[bnd[i]]): order[bnd[i]:bnd[i + 1]] for i in range(bnd.size - 1)}
            return dict(s=s, t=t, tk=tk, dc=dc, n=n, pos=[pmap.get(i, np.empty(0, int)) for i in range(n)])
        Pa, Pb = prep(a), prep(b)
        defs = {'pooled_spearman': self._pooled,
                'daily_cs_spearman': lambda P, idx=None: self._agg(self._daily(P, idx)),
                f'within_ticker_spearman_{self.agg}': self._within}
        rng = np.random.default_rng(self.seed)
        draws = {k: [] for k in defs}
        for _ in range(self.B):
            _, seq = self._sample(Pa, rng)
            ia = np.concatenate([Pa['pos'][c] for c in seq])
            ib = np.concatenate([Pb['pos'][c] for c in seq])
            for k, fn in defs.items():
                draws[k].append(fn(Pa, ia) - fn(Pb, ib))
        rows = []
        for k, fn in defs.items():
            pt = fn(Pa) - fn(Pb); arr = np.array(draws[k], float); arr = arr[np.isfinite(arr)]
            lo, hi = np.percentile(arr, [100 * self.alpha / 2, 100 * (1 - self.alpha / 2)])
            p = 2 * min((arr <= 0).mean(), (arr >= 0).mean())
            rows.append((k, pt, float(lo), float(hi), float(min(1, p)), la if pt > 0 else lb))
        return pd.DataFrame(rows, columns=['metric', 'diff', 'ci_low', 'ci_high', 'p_value', 'favours'])


# ---- classification: validation-tuned threshold + block-bootstrap CIs -------
def tune_threshold(y_true, score, grid=None):
    """Pick the probability threshold that maximises balanced accuracy on the
    (validation) data. Returns 0.5 if it cannot be estimated."""
    y = pd.to_numeric(pd.Series(y_true), errors='coerce').to_numpy(float)
    p = pd.to_numeric(pd.Series(score), errors='coerce').to_numpy(float)
    m = np.isfinite(y) & np.isfinite(p)
    y, p = y[m], p[m]
    if y.size == 0 or np.unique(y).size < 2:
        return 0.5
    grid = grid if grid is not None else np.quantile(p, np.linspace(0.05, 0.95, 19))
    best_t, best_s = 0.5, -1
    for t in grid:
        s = balanced_accuracy_score(y, (p >= t).astype(int))
        if s > best_s:
            best_s, best_t = s, float(t)
    return best_t

def _clf_point(y, p, thr):
    """Threshold-free (AUC, PR-AUC) + threshold-based (balanced acc, F1) metrics."""
    yhat = (p >= thr).astype(int)
    out = {}
    try: out['roc_auc'] = roc_auc_score(y, p)
    except Exception: out['roc_auc'] = np.nan
    try: out['pr_auc'] = average_precision_score(y, p)
    except Exception: out['pr_auc'] = np.nan
    out['balanced_acc'] = balanced_accuracy_score(y, yhat)
    out['f1'] = f1_score(y, yhat, zero_division=0)
    return out

def _block_boot_clf(d, thr, block=21, B=1000, alpha=0.05, seed=0):
    """Block-bootstrap over dates for the four classification metrics."""
    d = d.dropna(subset=[DATE_COL, SCORE_COL, TRUE_COL])
    dates = np.sort(d[DATE_COL].unique()); n = len(dates)
    pos = {dt: np.where(d[DATE_COL].values == dt)[0] for dt in dates}
    yv = pd.to_numeric(d[TRUE_COL], errors='coerce').to_numpy(float)
    pv = pd.to_numeric(d[SCORE_COL], errors='coerce').to_numpy(float)
    rng = np.random.default_rng(seed)
    keys = ['roc_auc', 'pr_auc', 'balanced_acc', 'f1']
    draws = {k: [] for k in keys}
    starts = np.arange(0, max(1, n - block + 1))
    for _ in range(B):
        seq = np.concatenate([np.arange(c, min(c + block, n))
                              for c in rng.choice(starts, size=int(np.ceil(n / block)), replace=True)])
        idx = np.concatenate([pos[dates[c]] for c in seq])
        y, p = yv[idx], pv[idx]
        if np.unique(y[np.isfinite(y)]).size < 2:
            continue
        m = _clf_point(y, p, thr)
        for k in keys:
            draws[k].append(m[k])
    res = {}
    for k in keys:
        arr = np.array(draws[k], float); arr = arr[np.isfinite(arr)]
        if arr.size:
            res[k] = (float(np.std(arr, ddof=1)),
                      *[float(v) for v in np.percentile(arr, [100 * alpha / 2, 100 * (1 - alpha / 2)])])
        else:
            res[k] = (np.nan, np.nan, np.nan)
    return res


# ---- drivers over a predictions dataframe -----------------------------------
def run_ranking_inference(pred, targets, split='test', challenger=None, benchmark=None,
                          B=1000, block=21, agg='median', maxlags=None):
    """Point + robust CI per model, and paired challenger-vs-benchmark, for the
    Spearman family. Use for CONTINUOUS targets only."""
    inf = SpearmanInference(B=B, block=block, agg=agg, maxlags=maxlags)
    out = []
    for tgt in targets:
        d = pred[(pred['target_name'] == tgt) & (pred[SPLIT_COL] == split)]
        for model in sorted(d['model'].dropna().unique()):
            s = inf.summary(d[d['model'] == model])
            s.insert(0, 'model', model); s.insert(0, 'analysis', 'summary'); s.insert(0, 'target', tgt)
            out.append(s)
        if challenger and benchmark:
            da, db = d[d['model'] == challenger], d[d['model'] == benchmark]
            if len(da) and len(db):
                c = inf.compare(da, db, challenger, benchmark)
                c = c.rename(columns={'diff': 'point'})
                c.insert(0, 'model', f'{challenger}-{benchmark}')
                c.insert(0, 'analysis', 'paired'); c.insert(0, 'target', tgt)
                out.append(c)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

def run_classification_inference(pred, targets, challenger=None, benchmark=None,
                                 B=1000, block=21):
    """Validation-tuned threshold, then test-set metrics with block-bootstrap CIs,
    per model; plus paired ROC-AUC difference. Use for BINARY targets only."""
    out = []
    for tgt in targets:
        dv_all = pred[(pred['target_name'] == tgt) & (pred[SPLIT_COL] == 'valid')]
        dt_all = pred[(pred['target_name'] == tgt) & (pred[SPLIT_COL] == 'test')]
        thr_by_model = {}
        for model in sorted(dt_all['model'].dropna().unique()):
            dv = dv_all[dv_all['model'] == model]; dt = dt_all[dt_all['model'] == model]
            if not len(dt):
                continue
            thr = tune_threshold(dv[TRUE_COL], dv[SCORE_COL]) if len(dv) else 0.5
            thr_by_model[model] = thr
            y = pd.to_numeric(dt[TRUE_COL], errors='coerce').to_numpy(float)
            p = pd.to_numeric(dt[SCORE_COL], errors='coerce').to_numpy(float)
            pt = _clf_point(y[np.isfinite(y)], p[np.isfinite(y)], thr)
            ci = _block_boot_clf(dt, thr, block=block, B=B)
            for metric in ['roc_auc', 'pr_auc', 'balanced_acc', 'f1']:
                se, lo, hi = ci[metric]
                out.append(dict(target=tgt, model=model, analysis='summary', metric=metric,
                                threshold=round(thr, 4), point=pt[metric], se=se,
                                ci_low=lo, ci_high=hi))
        # paired ROC-AUC (threshold-free) via block bootstrap over shared dates
        if challenger and benchmark and challenger in thr_by_model and benchmark in thr_by_model:
            da = dt_all[dt_all['model'] == challenger]; db = dt_all[dt_all['model'] == benchmark]
            common = np.intersect1d(da[DATE_COL].unique(), db[DATE_COL].unique())
            da, db = da[da[DATE_COL].isin(common)], db[db[DATE_COL].isin(common)]
            dates = np.sort(common); n = len(dates)
            pa = {dt: np.where(da[DATE_COL].values == dt)[0] for dt in dates}
            pb = {dt: np.where(db[DATE_COL].values == dt)[0] for dt in dates}
            ya = pd.to_numeric(da[TRUE_COL], errors='coerce').to_numpy(float)
            sa = pd.to_numeric(da[SCORE_COL], errors='coerce').to_numpy(float)
            yb = pd.to_numeric(db[TRUE_COL], errors='coerce').to_numpy(float)
            sb = pd.to_numeric(db[SCORE_COL], errors='coerce').to_numpy(float)
            rng = np.random.default_rng(0); starts = np.arange(0, max(1, n - block + 1)); diffs = []
            def _auc(y, s):
                m = np.isfinite(y) & np.isfinite(s)
                return roc_auc_score(y[m], s[m]) if np.unique(y[m]).size > 1 else np.nan
            for _ in range(B):
                seq = np.concatenate([np.arange(c, min(c + block, n))
                                      for c in rng.choice(starts, size=int(np.ceil(n / block)), replace=True)])
                ia = np.concatenate([pa[dates[c]] for c in seq]); ib = np.concatenate([pb[dates[c]] for c in seq])
                d = _auc(ya[ia], sa[ia]) - _auc(yb[ib], sb[ib])
                if np.isfinite(d):
                    diffs.append(d)
            arr = np.array(diffs, float)
            if arr.size:
                lo, hi = np.percentile(arr, [2.5, 97.5]); p = 2 * min((arr <= 0).mean(), (arr >= 0).mean())
                out.append(dict(target=tgt, model=f'{challenger}-{benchmark}', analysis='paired',
                                metric='roc_auc_diff', threshold=np.nan, point=float(arr.mean()),
                                se=float(arr.std(ddof=1)), ci_low=float(lo), ci_high=float(hi)))
    return pd.DataFrame(out)

## 8. Baseline experiment — fixed train / validation / test

Fits each model on the training split, evaluates on validation and test. Classification thresholds are tuned on validation in the inference layer (Section 7). Hyper-parameters are fixed at documented defaults.

*Applicable to:* all four selected targets.

In [16]:
EXPERIMENT_CONFIG = {
    'target_names': None,
    'feature_set_names': None,
    'model_names': None,
    'min_train_rows': 100,
    'min_valid_or_test_rows': 50,
    'save_row_level_predictions': True,
    'run_calibration': True,
    'run_daily_ranking': True,
    'run_topk': True,
    'run_within_ticker_ranking': True,
    'run_subgroup_diagnostics': True,
    'topk_percentages': [0.05, 0.10],
}


def should_run_name(name, selected_names):
    return selected_names is None or name in selected_names


def fit_predict_single_experiment(frames, target_name, target_cfg, feature_set_name, feature_cols, model_name, model, config=EXPERIMENT_CONFIG):
    target_col = target_cfg['column']
    task = target_cfg['task']

    train_df = filter_rows_for_target(frames['train'], target_name)
    valid_df = filter_rows_for_target(frames['valid'], target_name)
    test_df = filter_rows_for_target(frames['test'], target_name)

    X_train, y_train = prepare_xy(train_df, feature_cols, target_col, task)

    if len(y_train) < config['min_train_rows']:
        return [], [], {'status': 'skipped', 'reason': 'too_few_train_rows', 'n_train': len(y_train)}
    if task in ['binary', 'multiclass'] and y_train.nunique(dropna=True) < 2:
        return [], [], {'status': 'skipped', 'reason': 'single_class_train', 'n_train': len(y_train)}

    fitted_model = clone(model)
    fitted_model.fit(X_train, y_train)

    metric_rows = []
    prediction_frames = []

    for split_name, eval_df in [('valid', valid_df), ('test', test_df)]:
        X_eval, y_eval = prepare_xy(eval_df, feature_cols, target_col, task)
        if len(y_eval) < config['min_valid_or_test_rows']:
            continue

        y_pred = fitted_model.predict(X_eval)

        if task == 'binary':
            y_score = get_positive_proba(fitted_model, X_eval)
            if y_score is None:
                y_score = y_pred
            metrics = binary_metrics(y_eval, y_pred, y_score)
        elif task == 'regression':
            y_score = y_pred
            metrics = regression_metrics(y_eval, y_pred)
        elif task == 'multiclass':
            y_score = y_pred
            metrics = multiclass_metrics(y_eval, y_pred)
        else:
            raise ValueError(f'Unknown task: {task}')

        row = {
            'dataset_name': DATASET_CONFIG.get('dataset_name', ''),
            'target_name': target_name,
            'target_column': target_col,
            'task': task,
            'feature_set': feature_set_name,
            'n_features': len(feature_cols),
            'model': model_name,
            'split': split_name,
            'n_train_after_target_filter': len(y_train),
            'n_eval_after_target_filter': len(y_eval),
        }
        row.update(metrics)
        metric_rows.append(row)

        prediction_frames.append(
            build_prediction_frame(
                eval_df,
                y_eval,
                y_pred,
                y_score,
                target_name,
                task,
                feature_set_name,
                model_name,
                split_name,
            )
        )

    return metric_rows, prediction_frames, {'status': 'completed', 'n_train': len(y_train)}


def run_integrated_experiments(frames, feature_sets, target_configs=TARGET_CONFIGS, config=EXPERIMENT_CONFIG):
    all_metric_rows = []
    all_prediction_frames = []
    skipped_rows = []

    all_tmp = pd.concat(frames.values(), ignore_index=True, sort=False)
    available_targets = available_target_configs(all_tmp, target_configs)

    for target_name, target_cfg in available_targets.items():
        if not should_run_name(target_name, config['target_names']):
            continue

        models = get_models_for_task(target_cfg['task'])

        for feature_set_name, feature_cols in feature_sets.items():
            if not should_run_name(feature_set_name, config['feature_set_names']):
                continue
            if len(feature_cols) == 0:
                continue

            for model_name, model in models.items():
                if not should_run_name(model_name, config['model_names']):
                    continue

                print(f'Running: target={target_name} | features={feature_set_name} | model={model_name}')

                try:
                    metric_rows, prediction_frames, status = fit_predict_single_experiment(
                        frames,
                        target_name,
                        target_cfg,
                        feature_set_name,
                        feature_cols,
                        model_name,
                        model,
                        config,
                    )
                    all_metric_rows.extend(metric_rows)
                    all_prediction_frames.extend(prediction_frames)

                    if status['status'] != 'completed':
                        skipped_rows.append({'target_name': target_name, 'feature_set': feature_set_name, 'model': model_name, **status})

                except Exception as e:
                    skipped_rows.append({'target_name': target_name, 'feature_set': feature_set_name, 'model': model_name, 'status': 'error', 'reason': str(e)})
                    print('  -> skipped/error:', e)

    metrics_df = pd.DataFrame(all_metric_rows)
    predictions_df = pd.concat(all_prediction_frames, ignore_index=True, sort=False) if all_prediction_frames else pd.DataFrame()

    if len(predictions_df) > 0:
        predictions_df = add_signal_interface(predictions_df, target_configs=target_configs)

    skipped_df = pd.DataFrame(skipped_rows)

    register_table('model_metric_summary', metrics_df)
    register_table('skipped_or_failed_experiments', skipped_df)

    if config['save_row_level_predictions'] and len(predictions_df) > 0:
        _tgt_tag = 'signedlog' if any('signedlog' in str(_t) for _t in pd.Series(predictions_df['target_name']).dropna().unique()) else ''
        pred_path = PREDICTION_DIR / ((f'integrated_predictions_{RUN_TIMESTAMP}_{_tgt_tag}.parquet') if _tgt_tag else (f'integrated_predictions_{RUN_TIMESTAMP}.parquet'))
        predictions_df.to_parquet(pred_path, index=False)
        print('Prediction parquet exported:', pred_path)

    return metrics_df, predictions_df, skipped_df

#metrics_df, predictions_df, skipped_df = run_integrated_experiments(frames, FEATURE_SET_CONFIGS)

## 8.1 Simulator-compatible prediction export

In [17]:
# ============================================================
# Simulator-compatible prediction export
# ============================================================
# This section exports predictions in the structured format expected by
# downstream metric engines and trade simulation workflows.


def to_python_scalar(x):
    if pd.isna(x):
        return None
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, pd.Timestamp):
        return x.isoformat()
    return x


def get_timeframe_value(row):
    for col in ['attr__daily_timeframe', 'attr__timeframe', 'attr__report_timeframe']:
        if col in row and pd.notna(row[col]):
            return str(row[col])
    return 'd'


def build_prediction_result_record(row, include_actual=False):
    if INDEX_COL not in row or pd.isna(row[INDEX_COL]):
        raise ValueError(f'{INDEX_COL} is required for simulator-compatible prediction export.')

    signal_score = float(np.clip(row.get(SIGNAL_SCORE_COL, row.get(SCORE_COL, 0.0)), 0.0, 1.0))
    confidence = float(np.clip(row.get(CONFIDENCE_COL, 0.5), 0.0, 1.0))
    direction = str(row.get(DIRECTION_COL, 'long'))

    # Default long-side benchmark interface.
    trade_long = signal_score
    trade_short = 0.0
    trade_no_trade = 1.0 - max(trade_long, trade_short)

    if direction == 'short':
        trade_short = signal_score
        trade_long = 0.0
        trade_no_trade = 1.0 - max(trade_long, trade_short)

    long_payload = {'label': 'trade.long', 'prediction': trade_long, 'confidence': confidence}
    short_payload = {'label': 'trade.short', 'prediction': trade_short, 'confidence': confidence}
    no_trade_payload = {'label': 'trade.no_trade', 'prediction': trade_no_trade, 'confidence': confidence}

    if include_actual:
        actual = to_python_scalar(row.get(TRUE_COL))
        long_payload['actual_value'] = actual

    timestamp_value = row.get(DATE_COL)
    if isinstance(timestamp_value, pd.Timestamp):
        timestamp_value = timestamp_value.isoformat()
    else:
        timestamp_value = str(timestamp_value)

    return {
        'index': int(row[INDEX_COL]),
        'ticker': str(row[TICKER_COL]),
        'timestamp': timestamp_value,
        'timeframe': get_timeframe_value(row),
        'predictions': {
            'trade.long': long_payload,
            'trade.short': short_payload,
            'trade.no_trade': no_trade_payload,
            'signal.score': {'label': 'signal_score', 'prediction': signal_score, 'confidence': confidence},
        },
    }


def export_prediction_submission_json(
    predictions_df,
    metrics_df,
    experiment_id,
    split,
    target_name,
    feature_set,
    model_name,
    output_root=PREDICTION_DIR,
    split_file=None,
    model_type=None,
    include_actual=False,
    export_jsonl=False,
):
    """
    Export one structured prediction file for one target-feature-model-split combination.
    The JSON file follows the required top-level structure:
    metadata, prediction_results, and model_metrics.
    """
    if predictions_df is None or len(predictions_df) == 0:
        return None
    if INDEX_COL not in predictions_df.columns:
        raise ValueError(f'{INDEX_COL} is missing from predictions_df. Re-run experiments with the updated build_prediction_frame().')

    df = predictions_df[
        (predictions_df['target_name'] == target_name)
        & (predictions_df['feature_set'] == feature_set)
        & (predictions_df['model'] == model_name)
        & (predictions_df[SPLIT_COL] == split)
    ].copy()

    if len(df) == 0:
        return None

    sort_cols = [c for c in [DATE_COL, TICKER_COL, INDEX_COL] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols)

    model_id = safe_table_name(f'{model_name}_{feature_set}_{target_name}')
    if model_type is None:
        model_type = model_name

    metadata = {
        'prediction_schema_version': '1.0',
        'experiment_id': experiment_id,
        'split': split,
        'model_id': model_id,
        'model_type': model_type,
        'created_at': pd.Timestamp.utcnow().isoformat(),
        'target_name': target_name,
        'feature_set': feature_set,
        'training_dataset': DATASET_CONFIG.get('dataset_name', ''),
        'notes': 'Generated by Integrated_Experiment_Runner. signal_score is rank-normalised within target-feature-model-split; confidence is a documented distance-from-neutral proxy.',
    }

    if split_file is not None:
        metadata['split_file'] = split_file

    model_metrics = {}
    if metrics_df is not None and len(metrics_df) > 0:
        m = metrics_df[
            (metrics_df['target_name'] == target_name)
            & (metrics_df['feature_set'] == feature_set)
            & (metrics_df['model'] == model_name)
            & (metrics_df[SPLIT_COL] == split)
        ]
        if len(m) > 0:
            model_metrics = {k: to_python_scalar(v) for k, v in m.iloc[0].to_dict().items()}

    prediction_results = [build_prediction_result_record(row, include_actual=include_actual) for _, row in df.iterrows()]

    payload = {
        'metadata': metadata,
        'prediction_results': prediction_results,
        'model_metrics': model_metrics,
    }

    out_dir = Path(output_root) / safe_table_name(experiment_id).lower()
    out_dir.mkdir(parents=True, exist_ok=True)

    json_path = out_dir / f'{model_id}_{split}_predictions.json'
    with json_path.open('w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    output_paths = [json_path]

    if export_jsonl:
        jsonl_path = out_dir / f'{model_id}_{split}_predictions.jsonl'
        with jsonl_path.open('w', encoding='utf-8') as f:
            f.write(json.dumps({'section': 'metadata', 'metadata': metadata}, ensure_ascii=False) + '\n')
            for rec in prediction_results:
                f.write(json.dumps({'section': 'prediction_result', 'data': rec}, ensure_ascii=False) + '\n')
            f.write(json.dumps({'section': 'model_metrics', 'model_metrics': model_metrics}, ensure_ascii=False) + '\n')
        output_paths.append(jsonl_path)

    return output_paths


def export_all_prediction_submissions(
    predictions_df,
    metrics_df,
    experiment_id=None,
    output_root=PREDICTION_DIR,
    split='test',
    include_actual=False,
    export_jsonl=False,
):
    """
    Export simulator-compatible files for all target-feature-model combinations in one split.
    """
    if experiment_id is None:
        experiment_id = DATASET_CONFIG.get('experiment_id', DATASET_CONFIG.get('dataset_name', 'experiment'))

    paths = []
    if predictions_df is None or len(predictions_df) == 0:
        return paths

    group_cols = ['target_name', 'feature_set', 'model', SPLIT_COL]
    for (target_name, feature_set, model_name, split_name), _ in predictions_df.groupby(group_cols, dropna=False):
        if split_name != split:
            continue
        out = export_prediction_submission_json(
            predictions_df=predictions_df,
            metrics_df=metrics_df,
            experiment_id=experiment_id,
            split=split_name,
            target_name=target_name,
            feature_set=feature_set,
            model_name=model_name,
            output_root=output_root,
            include_actual=include_actual,
            export_jsonl=export_jsonl,
        )
        if out:
            paths.extend(out)

    submission_index = pd.DataFrame({'prediction_file': [str(p) for p in paths]})
    register_table('prediction_submission_files', submission_index)
    print(f'Exported {len(paths)} prediction submission files.')
    return paths

# Example after running experiments:
# prediction_submission_paths = export_all_prediction_submissions(
#     predictions_df=predictions_df,
#     metrics_df=metrics_df,
#     experiment_id=DATASET_CONFIG.get('dataset_name'),
#     split='test',
#     include_actual=False,
#     export_jsonl=True,
# )

## 9. Post-model diagnostics (ranking, Top-K, subgroups)

In [18]:

# diagnostic_tables = run_prediction_diagnostics(predictions_df)

## 10. Best-model comparison tables

In [19]:

# best_model_tables = build_best_model_tables(metrics_df)
# best_ranking_tables = build_best_ranking_tables()

## 11. Current-PnL temporal decile analysis

*Applicable label:* `rl_long_current_pnl` only.

In [20]:
def predicted_decile_outcome_table(pred_df, n_deciles=10, min_rows=30):
    if pred_df is None or len(pred_df) < min_rows: return pd.DataFrame()
    df = pred_df.dropna(subset=[TRUE_COL, SCORE_COL]).copy()
    if len(df) < min_rows: return pd.DataFrame()
    df['prediction_rank_pct'] = df[SCORE_COL].rank(method='first', pct=True)
    df['prediction_decile'] = np.ceil(df['prediction_rank_pct'] * n_deciles).clip(1, n_deciles).astype(int)
    return df.groupby('prediction_decile').agg(n=(TRUE_COL, 'size'), mean_true=(TRUE_COL, 'mean'), median_true=(TRUE_COL, 'median'), std_true=(TRUE_COL, 'std'), mean_score=(SCORE_COL, 'mean'), positive_true_rate=(TRUE_COL, lambda x: (pd.to_numeric(x, errors='coerce') > 0).mean()), negative_true_rate=(TRUE_COL, lambda x: (pd.to_numeric(x, errors='coerce') < 0).mean())).reset_index()


def temporal_decile_analysis(predictions_df, target_name='rl_long_current_pnl', n_deciles=10):
    if predictions_df is None or len(predictions_df) == 0: return pd.DataFrame(), pd.DataFrame()
    df = predictions_df[predictions_df['target_name'] == target_name].copy()
    if len(df) == 0: return pd.DataFrame(), pd.DataFrame()
    group_cols = ['target_name', 'feature_set', 'model', SPLIT_COL]
    detail_parts = []; year_parts = []
    for keys, g in df.groupby(group_cols, dropna=False):
        detail = predicted_decile_outcome_table(g, n_deciles=n_deciles)
        if len(detail):
            for col, value in zip(group_cols, keys): detail[col] = value
            detail_parts.append(detail)
        if YEAR_COL in g.columns:
            for year, gy in g.groupby(YEAR_COL, dropna=False):
                yd = predicted_decile_outcome_table(gy, n_deciles=n_deciles, min_rows=20)
                if len(yd):
                    for col, value in zip(group_cols, keys): yd[col] = value
                    yd[YEAR_COL] = year; year_parts.append(yd)
    detail_df = pd.concat(detail_parts, ignore_index=True, sort=False) if detail_parts else pd.DataFrame()
    year_df = pd.concat(year_parts, ignore_index=True, sort=False) if year_parts else pd.DataFrame()
    register_table('current_pnl_prediction_deciles', detail_df)
    register_table('current_pnl_prediction_deciles_by_year', year_df)
    return detail_df, year_df

# current_pnl_deciles, current_pnl_deciles_by_year = temporal_decile_analysis(predictions_df)

## 12. Yearly walk-forward — expanding & rolling (2-year start)

Expanding and rolling both begin from a **2-year** minimum training window and rotate yearly. With 2020–2026 data this yields test folds **2022–2026** (2026 partial). Also builds the leakage-safe within-ticker score percentile for normalised cross-sectional ranking.
Rolling 1yr and 3 yr settings are trained in another notebook due to computation concerns.

In [21]:
# ============================================================
# Yearly walk-forward experiments
# ============================================================
# This section adds two yearly retraining designs:
# 1. Expanding window: train on all available years before the test year.
# 2. Rolling 2-year window: train only on the most recent two years before the test year.
#
# Both designs evaluate one out-of-sample year at a time. For each fitted model,
# the notebook also maps raw prediction scores into historical within-ticker
# percentiles using the training-window score distribution only. This avoids
# using future/test score distributions when creating ticker-normalised scores.

WALK_FORWARD_CONFIG = {
    'enabled': True,
    'schemes': ['expanding', 'rolling_2y'],
    'retrain_frequency': 'yearly',
    'rolling_window_years': 2,
    'min_train_years': 2,
    'start_test_year': None,   # Set to an integer, e.g. 2021, to restrict folds.
    'end_test_year': None,     # Set to an integer, e.g. 2026, to restrict folds.
    'save_row_level_predictions': True,
}

WITHIN_TICKER_PERCENTILE_SCORE_COL = 'within_ticker_score_percentile'
GLOBAL_PERCENTILE_FALLBACK_COL = 'global_train_score_percentile'


def get_year_values(df):
    """Return a sorted list of usable calendar years."""
    if YEAR_COL not in df.columns:
        if DATE_COL not in df.columns:
            raise ValueError(f'Cannot build walk-forward folds: missing both {YEAR_COL} and {DATE_COL}.')
        df = df.copy()
        df[YEAR_COL] = pd.to_datetime(df[DATE_COL], errors='coerce').dt.year
    years = pd.Series(df[YEAR_COL]).dropna().astype(int).sort_values().unique().tolist()
    return years


def build_yearly_walk_forward_folds(all_data, scheme='expanding', rolling_window_years=2, min_train_years=2, start_test_year=None, end_test_year=None):
    """
    Build yearly out-of-sample folds.

    Expanding: train_years = all years < test_year.
    Rolling:   train_years = the last `rolling_window_years` years before test_year.
    """
    years = get_year_values(all_data)
    folds = []

    for test_year in years:
        if start_test_year is not None and test_year < int(start_test_year):
            continue
        if end_test_year is not None and test_year > int(end_test_year):
            continue

        prior_years = [y for y in years if y < test_year]
        if scheme == 'expanding':
            train_years = prior_years
            window_label = 'expanding'
        elif scheme == 'rolling_2y':
            train_years = prior_years[-int(rolling_window_years):]
            window_label = f'rolling_{rolling_window_years}y'
        else:
            raise ValueError(f'Unknown walk-forward scheme: {scheme}')

        if len(train_years) < min_train_years:
            continue
        if test_year not in years:
            continue

        folds.append({
            'walkforward_scheme': scheme,
            'window_label': window_label,
            'retrain_frequency': 'yearly',
            'fold_id': f'{window_label}_test_{test_year}',
            'test_year': int(test_year),
            'train_start_year': int(min(train_years)),
            'train_end_year': int(max(train_years)),
            'n_train_years': int(len(train_years)),
            'train_years': train_years,
        })

    return folds


def historical_percentile_from_train_distribution(eval_scores, train_scores):
    """
    Map evaluation scores to percentiles using the training score distribution only.
    The output is the share of historical training scores less than or equal to
    the evaluation score. This is point-in-time safe for walk-forward evaluation.
    """
    train_scores = pd.to_numeric(pd.Series(train_scores), errors='coerce').dropna().to_numpy(dtype=float)
    eval_scores = pd.to_numeric(pd.Series(eval_scores), errors='coerce')

    if len(train_scores) == 0:
        return pd.Series(np.nan, index=eval_scores.index)

    train_scores = np.sort(train_scores)
    out = []
    for value in eval_scores:
        if pd.isna(value):
            out.append(np.nan)
        else:
            out.append(np.searchsorted(train_scores, float(value), side='right') / len(train_scores))
    return pd.Series(out, index=eval_scores.index, dtype=float)


def add_historical_within_ticker_percentiles(eval_pred_df, train_score_df, score_col=SCORE_COL):
    """
    Add ticker-normalised score percentiles to evaluation predictions.

    For each ticker, the percentile is computed against that ticker's training-window
    predicted score distribution. If a ticker has no usable training score history,
    the function falls back to the global training score distribution.
    """
    if eval_pred_df is None or len(eval_pred_df) == 0:
        return eval_pred_df

    out = eval_pred_df.copy()
    if TICKER_COL not in out.columns or TICKER_COL not in train_score_df.columns:
        out[WITHIN_TICKER_PERCENTILE_SCORE_COL] = historical_percentile_from_train_distribution(out[score_col], train_score_df[score_col]).values
        out[GLOBAL_PERCENTILE_FALLBACK_COL] = True
        return out

    global_pct = historical_percentile_from_train_distribution(out[score_col], train_score_df[score_col])
    out[WITHIN_TICKER_PERCENTILE_SCORE_COL] = np.nan
    out[GLOBAL_PERCENTILE_FALLBACK_COL] = False

    train_by_ticker = {
        ticker: g[score_col]
        for ticker, g in train_score_df.dropna(subset=[score_col]).groupby(TICKER_COL, dropna=False)
    }

    for ticker, idx in out.groupby(TICKER_COL, dropna=False).groups.items():
        idx = list(idx)
        if ticker in train_by_ticker and pd.Series(train_by_ticker[ticker]).dropna().shape[0] > 0:
            pct = historical_percentile_from_train_distribution(out.loc[idx, score_col], train_by_ticker[ticker])
            out.loc[idx, WITHIN_TICKER_PERCENTILE_SCORE_COL] = pct.values
        else:
            out.loc[idx, WITHIN_TICKER_PERCENTILE_SCORE_COL] = global_pct.loc[idx].values
            out.loc[idx, GLOBAL_PERCENTILE_FALLBACK_COL] = True

    return out


def score_model_for_task(fitted_model, X, task):
    """Return prediction labels and ranking scores for a fitted model."""
    y_pred = fitted_model.predict(X)
    if task == 'binary':
        y_score = get_positive_proba(fitted_model, X)
        if y_score is None:
            y_score = y_pred
    elif task == 'regression':
        y_score = y_pred
    elif task == 'multiclass':
        # Multiclass is kept as predicted class by default because there is no single
        # universally meaningful positive probability across action labels.
        y_score = y_pred
    else:
        raise ValueError(f'Unknown task: {task}')
    return y_pred, y_score


def metric_for_task(task, y_true, y_pred, y_score=None):
    if task == 'binary':
        return binary_metrics(y_true, y_pred, y_score)
    if task == 'regression':
        return regression_metrics(y_true, y_pred)
    if task == 'multiclass':
        return multiclass_metrics(y_true, y_pred)
    raise ValueError(f'Unknown task: {task}')


def fit_predict_single_walk_forward_fold(all_data, fold, target_name, target_cfg, feature_set_name, feature_cols, model_name, model, config=EXPERIMENT_CONFIG):
    """Fit one model on one walk-forward fold and evaluate one test year."""
    target_col = target_cfg['column']
    task = target_cfg['task']

    train_df = all_data[all_data[YEAR_COL].isin(fold['train_years'])].copy()
    eval_df = all_data[all_data[YEAR_COL] == fold['test_year']].copy()

    train_df = filter_rows_for_target(train_df, target_name)
    eval_df = filter_rows_for_target(eval_df, target_name)

    X_train, y_train = prepare_xy(train_df, feature_cols, target_col, task)
    if len(y_train) < config['min_train_rows']:
        return [], [], {'status': 'skipped', 'reason': 'too_few_train_rows', 'n_train': len(y_train)}
    if task in ['binary', 'multiclass'] and y_train.nunique(dropna=True) < 2:
        return [], [], {'status': 'skipped', 'reason': 'single_class_train', 'n_train': len(y_train)}

    X_eval, y_eval = prepare_xy(eval_df, feature_cols, target_col, task)
    if len(y_eval) < config['min_valid_or_test_rows']:
        return [], [], {'status': 'skipped', 'reason': 'too_few_eval_rows', 'n_eval': len(y_eval)}

    fitted_model = clone(model)
    fitted_model.fit(X_train, y_train)

    y_pred_eval, y_score_eval = score_model_for_task(fitted_model, X_eval, task)
    y_pred_train, y_score_train = score_model_for_task(fitted_model, X_train, task)

    metrics = metric_for_task(task, y_eval, y_pred_eval, y_score_eval)
    row = {
        'dataset_name': DATASET_CONFIG.get('dataset_name', ''),
        'target_name': target_name,
        'target_column': target_col,
        'task': task,
        'feature_set': feature_set_name,
        'n_features': len(feature_cols),
        'model': model_name,
        SPLIT_COL: 'walkforward_test',
        'walkforward_scheme': fold['walkforward_scheme'],
        'window_label': fold['window_label'],
        'retrain_frequency': fold['retrain_frequency'],
        'fold_id': fold['fold_id'],
        'test_year': fold['test_year'],
        'train_start_year': fold['train_start_year'],
        'train_end_year': fold['train_end_year'],
        'n_train_years': fold['n_train_years'],
        'n_train_after_target_filter': len(y_train),
        'n_eval_after_target_filter': len(y_eval),
    }
    row.update(metrics)

    eval_pred = build_prediction_frame(
        eval_df,
        y_eval,
        y_pred_eval,
        y_score_eval,
        target_name,
        task,
        feature_set_name,
        model_name,
        split_name='walkforward_test',
    )

    train_score_frame = build_prediction_frame(
        train_df,
        y_train,
        y_pred_train,
        y_score_train,
        target_name,
        task,
        feature_set_name,
        model_name,
        split_name='walkforward_train_history',
    )

    eval_pred = add_historical_within_ticker_percentiles(eval_pred, train_score_frame, score_col=SCORE_COL)

    for key in ['walkforward_scheme', 'window_label', 'retrain_frequency', 'fold_id', 'test_year', 'train_start_year', 'train_end_year', 'n_train_years']:
        eval_pred[key] = fold[key]

    return [row], [eval_pred], {'status': 'completed', 'n_train': len(y_train), 'n_eval': len(y_eval)}


def run_walk_forward_experiments(frames, feature_sets, target_configs=TARGET_CONFIGS, config=EXPERIMENT_CONFIG, walk_config=WALK_FORWARD_CONFIG):
    """
    Run expanding-window and rolling-window yearly retraining experiments.
    The original static-split runner is left unchanged; this function produces a
    separate walk-forward metrics table and row-level prediction table.
    """
    all_metric_rows = []
    all_prediction_frames = []
    skipped_rows = []

    all_data = pd.concat(frames.values(), ignore_index=True, sort=False).copy()
    if DATE_COL in all_data.columns:
        all_data[DATE_COL] = pd.to_datetime(all_data[DATE_COL], errors='coerce')
    if YEAR_COL not in all_data.columns or all_data[YEAR_COL].isna().all():
        all_data[YEAR_COL] = all_data[DATE_COL].dt.year
    all_data[YEAR_COL] = pd.to_numeric(all_data[YEAR_COL], errors='coerce')

    available_targets = available_target_configs(all_data, target_configs)
    all_folds = []
    for scheme in walk_config.get('schemes', ['expanding']):
        folds = build_yearly_walk_forward_folds(
            all_data,
            scheme=scheme,
            rolling_window_years=walk_config.get('rolling_window_years', 2),
            min_train_years=walk_config.get('min_train_years', 2),
            start_test_year=walk_config.get('start_test_year'),
            end_test_year=walk_config.get('end_test_year'),
        )
        all_folds.extend(folds)

    fold_table = pd.DataFrame(all_folds)
    register_table('walk_forward_fold_definition', fold_table)
    print(f'Walk-forward folds: {len(all_folds)}')
    if len(fold_table):
        display(fold_table[['walkforward_scheme', 'fold_id', 'train_start_year', 'train_end_year', 'n_train_years', 'test_year']].head(20))

    for target_name, target_cfg in available_targets.items():
        if not should_run_name(target_name, config['target_names']):
            continue

        models = get_models_for_task(target_cfg['task'])
        for feature_set_name, feature_cols in feature_sets.items():
            if not should_run_name(feature_set_name, config['feature_set_names']):
                continue
            if len(feature_cols) == 0:
                continue

            for model_name, model in models.items():
                if not should_run_name(model_name, config['model_names']):
                    continue

                for fold in all_folds:
                    print(f"WF: scheme={fold['window_label']} | test_year={fold['test_year']} | target={target_name} | features={feature_set_name} | model={model_name}")
                    try:
                        metric_rows, prediction_frames, status = fit_predict_single_walk_forward_fold(
                            all_data,
                            fold,
                            target_name,
                            target_cfg,
                            feature_set_name,
                            feature_cols,
                            model_name,
                            model,
                            config,
                        )
                        all_metric_rows.extend(metric_rows)
                        all_prediction_frames.extend(prediction_frames)
                        if status['status'] != 'completed':
                            skipped_rows.append({
                                'target_name': target_name,
                                'feature_set': feature_set_name,
                                'model': model_name,
                                **{k: fold[k] for k in ['walkforward_scheme', 'fold_id', 'test_year', 'train_start_year', 'train_end_year', 'n_train_years']},
                                **status,
                            })
                    except Exception as e:
                        skipped_rows.append({
                            'target_name': target_name,
                            'feature_set': feature_set_name,
                            'model': model_name,
                            **{k: fold[k] for k in ['walkforward_scheme', 'fold_id', 'test_year', 'train_start_year', 'train_end_year', 'n_train_years']},
                            'status': 'error',
                            'reason': str(e),
                        })
                        print('  -> skipped/error:', e)

    metrics_df = pd.DataFrame(all_metric_rows)
    predictions_df = pd.concat(all_prediction_frames, ignore_index=True, sort=False) if all_prediction_frames else pd.DataFrame()
    if len(predictions_df) > 0:
        predictions_df = add_signal_interface(predictions_df, target_configs=target_configs)

    skipped_df = pd.DataFrame(skipped_rows)

    register_table('walk_forward_model_metric_summary', metrics_df)
    register_table('walk_forward_skipped_or_failed_experiments', skipped_df)

    if walk_config.get('save_row_level_predictions', True) and len(predictions_df) > 0:
        _tgt_tag = 'signedlog' if any('signedlog' in str(_t) for _t in pd.Series(predictions_df['target_name']).dropna().unique()) else ''
        pred_path = PREDICTION_DIR / ((f'walk_forward_predictions_{RUN_TIMESTAMP}_{_tgt_tag}.parquet') if _tgt_tag else (f'walk_forward_predictions_{RUN_TIMESTAMP}.parquet'))
        predictions_df.to_parquet(pred_path, index=False)
        print('Walk-forward prediction parquet exported:', pred_path)

    return metrics_df, predictions_df, skipped_df


def daily_cross_sectional_spearman_for_score(pred_df, score_col, output_col='daily_spearman', min_daily_rows=5):
    """Daily cross-sectional Spearman using a selected ranking score column."""
    rows = []
    if DATE_COL not in pred_df.columns or score_col not in pred_df.columns:
        return pd.DataFrame()
    for date, g in pred_df.groupby(DATE_COL):
        g = g.dropna(subset=[TRUE_COL, score_col]).copy()
        if len(g) < min_daily_rows:
            continue
        rows.append({
            DATE_COL: date,
            'n': len(g),
            output_col: safe_spearman(g[TRUE_COL], g[score_col]),
            'mean_true': pd.to_numeric(g[TRUE_COL], errors='coerce').mean(),
            f'mean_{score_col}': pd.to_numeric(g[score_col], errors='coerce').mean(),
            'global_percentile_fallback_rate': g[GLOBAL_PERCENTILE_FALLBACK_COL].mean() if GLOBAL_PERCENTILE_FALLBACK_COL in g.columns else np.nan,
        })
    return pd.DataFrame(rows)


def run_within_ticker_percentile_diagnostics(predictions_df, min_daily_rows=5):
    """
    Evaluate whether ticker-normalised prediction percentiles improve daily
    cross-sectional ranking. The score used here is:
    within_ticker_score_percentile = today's score percentile relative to that
    ticker's training-window score distribution.
    """
    if predictions_df is None or len(predictions_df) == 0 or WITHIN_TICKER_PERCENTILE_SCORE_COL not in predictions_df.columns:
        print('No within-ticker percentile score available for diagnostics.')
        return {}

    diagnostic_tables = {}
    base_group_cols = ['target_name', 'task', 'feature_set', 'model', SPLIT_COL]
    extra_group_cols = [c for c in ['walkforward_scheme', 'window_label', 'retrain_frequency', 'fold_id', 'test_year'] if c in predictions_df.columns]
    group_cols = base_group_cols + extra_group_cols

    parts = []
    for keys, g in predictions_df.groupby(group_cols, dropna=False):
        one = daily_cross_sectional_spearman_for_score(
            g,
            score_col=WITHIN_TICKER_PERCENTILE_SCORE_COL,
            output_col='daily_spearman_within_ticker_percentile',
            min_daily_rows=min_daily_rows,
        )
        if len(one) == 0:
            continue
        for col, value in zip(group_cols, keys):
            one[col] = value
        parts.append(one)

    detail = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
    if len(detail):
        summary_cols = base_group_cols + [c for c in ['walkforward_scheme', 'window_label', 'retrain_frequency', 'test_year'] if c in detail.columns]
        summary = detail.groupby(summary_cols, dropna=False).agg(
            n_days=('daily_spearman_within_ticker_percentile', 'count'),
            mean_daily_spearman_within_ticker_percentile=('daily_spearman_within_ticker_percentile', 'mean'),
            median_daily_spearman_within_ticker_percentile=('daily_spearman_within_ticker_percentile', 'median'),
            positive_spearman_day_rate=('daily_spearman_within_ticker_percentile', lambda x: (x > 0).mean()),
            mean_global_percentile_fallback_rate=('global_percentile_fallback_rate', 'mean'),
        ).reset_index()
    else:
        summary = pd.DataFrame()

    diagnostic_tables['daily_spearman_within_ticker_percentile_detail'] = register_table('daily_spearman_within_ticker_percentile_detail', detail)
    diagnostic_tables['daily_spearman_within_ticker_percentile_summary'] = register_table('daily_spearman_within_ticker_percentile_summary', summary)
    return diagnostic_tables

In [ ]:

# ============================================================
## Walk-forward metric and grouping helpers
# ============================================================
#
# 1) The original diagnostic grouping used only target/task/feature/model/split.
#    In walk-forward output, split is always "walkforward_test", so expanding
#    and rolling folds were aggregated together.
# 2) The original metric helpers converted y_true and y_pred separately to
#    pandas Series. When y_true kept original row indices but predictions used
#    RangeIndex, pandas aligned by index and could leave no matched rows.
#    These replacement helpers evaluate positionally and therefore preserve
#    the intended y/pred pairing.


def _as_positional_series(x, dtype=None):
    s = pd.Series(np.asarray(x))
    if dtype is not None:
        s = s.astype(dtype)
    return s


def safe_spearman(y_true, y_score):
    y_true = _as_positional_series(y_true)
    y_score = _as_positional_series(y_score)
    valid = y_true.notna() & y_score.notna()
    if valid.sum() < 3 or y_true.loc[valid].nunique() < 2 or y_score.loc[valid].nunique() < 2:
        return np.nan
    return y_true.loc[valid].corr(y_score.loc[valid], method='spearman')


def safe_pearson(y_true, y_score):
    y_true = _as_positional_series(y_true)
    y_score = _as_positional_series(y_score)
    valid = y_true.notna() & y_score.notna()
    if valid.sum() < 3 or y_true.loc[valid].nunique() < 2 or y_score.loc[valid].nunique() < 2:
        return np.nan
    return y_true.loc[valid].corr(y_score.loc[valid], method='pearson')


def regression_metrics(y_true, y_pred):
    y_true = _as_positional_series(y_true, float)
    y_pred = _as_positional_series(y_pred, float)
    valid = y_true.notna() & y_pred.notna()
    if valid.sum() == 0:
        return {}

    yt = y_true.loc[valid]
    yp = y_pred.loc[valid]
    errors = yp - yt
    abs_errors = errors.abs()
    denom = yt.abs().sum()

    directional_accuracy = np.nan
    if yt.nunique() > 1 and yp.nunique() > 1:
        directional_accuracy = (np.sign(yt) == np.sign(yp)).mean()

    return {
        'n': int(valid.sum()),
        'mae': mean_absolute_error(yt, yp),
        'mse': mean_squared_error(yt, yp),
        'rmse': np.sqrt(mean_squared_error(yt, yp)),
        'r2': r2_score(yt, yp) if yt.nunique() > 1 else np.nan,
        'explained_variance': explained_variance_score(yt, yp) if yt.nunique() > 1 else np.nan,
        'pearson': safe_pearson(yt, yp),
        'spearman': safe_spearman(yt, yp),
        'information_coefficient': safe_pearson(yt, yp),
        'rank_information_coefficient': safe_spearman(yt, yp),
        'directional_accuracy': directional_accuracy,
        'weighted_mape': abs_errors.sum() / denom if denom > 0 else np.nan,
        'error_p50': abs_errors.quantile(0.50),
        'error_p90': abs_errors.quantile(0.90),
        'error_p95': abs_errors.quantile(0.95),
        'error_p99': abs_errors.quantile(0.99),
        'mean_y_true': yt.mean(),
        'mean_y_pred': yp.mean(),
        'std_y_true': yt.std(),
        'std_y_pred': yp.std(),
    }


def binary_metrics(y_true, y_pred, y_score=None):
    y_true = _as_positional_series(y_true)
    y_pred = _as_positional_series(y_pred)
    valid = y_true.notna() & y_pred.notna()
    if y_score is not None:
        y_score = _as_positional_series(y_score, float)
        valid = valid & y_score.notna()

    if valid.sum() == 0:
        return {}

    yt = y_true.loc[valid].astype(int)
    yp = y_pred.loc[valid].astype(int)

    try:
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    except Exception:
        tn = fp = fn = tp = np.nan

    out = {
        'n': int(valid.sum()),
        'accuracy': accuracy_score(yt, yp),
        'balanced_accuracy': balanced_accuracy_score(yt, yp) if yt.nunique() > 1 else np.nan,
        'precision': precision_score(yt, yp, zero_division=0),
        'recall': recall_score(yt, yp, zero_division=0),
        'f1': f1_score(yt, yp, zero_division=0),
        'macro_f1': f1_score(yt, yp, average='macro', zero_division=0),
        'matthews_corrcoef': matthews_corrcoef(yt, yp) if yt.nunique() > 1 else np.nan,
        'cohen_kappa': cohen_kappa_score(yt, yp) if yt.nunique() > 1 else np.nan,
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'tp': tp,
        'positive_rate_true': yt.mean(),
        'positive_rate_pred': yp.mean(),
    }

    if y_score is not None:
        ys = y_score.loc[valid].astype(float)
        out['mean_predicted_probability'] = ys.mean()
        if yt.nunique() > 1 and ys.nunique() > 1:
            ys_prob = np.clip(ys, 1e-6, 1 - 1e-6)
            out.update({
                'roc_auc': roc_auc_score(yt, ys),
                'pr_auc': average_precision_score(yt, ys),
                'brier_score': brier_score_loss(yt, np.clip(ys, 0, 1)),
                'log_loss': log_loss(yt, ys_prob),
                'spearman': safe_spearman(yt, ys),
            })

    return out


def multiclass_metrics(y_true, y_pred, y_proba=None):
    y_true = _as_positional_series(y_true)
    y_pred = _as_positional_series(y_pred)
    valid = y_true.notna() & y_pred.notna()
    if valid.sum() == 0:
        return {}

    yt = y_true.loc[valid]
    yp = y_pred.loc[valid]
    out = {
        'n': int(valid.sum()),
        'accuracy': accuracy_score(yt, yp),
        'balanced_accuracy': balanced_accuracy_score(yt, yp) if yt.nunique() > 1 else np.nan,
        'macro_f1': f1_score(yt, yp, average='macro', zero_division=0),
        'weighted_f1': f1_score(yt, yp, average='weighted', zero_division=0),
        'n_classes_true': yt.nunique(),
        'n_classes_pred': yp.nunique(),
    }

    try:
        out['spearman_class_rank'] = safe_spearman(pd.to_numeric(yt), pd.to_numeric(yp))
    except Exception:
        out['spearman_class_rank'] = np.nan

    return out


def add_signal_interface(predictions_df, target_configs=TARGET_CONFIGS):
    """
    Convert raw model outputs into signal_score, direction, and confidence.
    For walk-forward predictions, rank-normalise within each scheme/fold/model group
    rather than across expanding and rolling outputs together.
    """
    if predictions_df is None or len(predictions_df) == 0:
        return predictions_df

    df = predictions_df.copy()
    base_cols = ['target_name', 'feature_set', 'model', SPLIT_COL]
    extra_cols = [c for c in ['walkforward_scheme', 'window_label', 'fold_id', 'test_year'] if c in df.columns]
    group_cols = base_cols + extra_cols
    parts = []

    for _, g in df.groupby(group_cols, dropna=False):
        g = g.copy()
        target_name = g['target_name'].iloc[0]
        target_cfg = target_configs.get(target_name, {})
        direction_type = target_cfg.get('direction', 'higher_is_better')

        raw_score = pd.to_numeric(g[SCORE_COL], errors='coerce')
        rank_pct = raw_score.rank(method='average', pct=True)
        signal_score = 1 - rank_pct if direction_type == 'lower_is_better' else rank_pct
        signal_score = signal_score.clip(0, 1)

        g[SIGNAL_SCORE_COL] = signal_score
        g[CONFIDENCE_COL] = ((signal_score - 0.5).abs() * 2).clip(0.01, 1.0)
        g[DIRECTION_COL] = np.where(signal_score >= 0.5, 'long', 'no_trade')

        if direction_type == 'action':
            action_map = {0: 'no_trade', 1: 'long', 2: 'short'}
            numeric_pred = pd.to_numeric(g[PRED_COL], errors='coerce')
            mapped = numeric_pred.map(action_map)
            g[DIRECTION_COL] = mapped.fillna(g[DIRECTION_COL])

        parts.append(g)

    return pd.concat(parts, ignore_index=True, sort=False)


def get_prediction_group_cols(predictions_df, include_fold=False):
    base_cols = ['target_name', 'task', 'feature_set', 'model', SPLIT_COL]
    wf_cols = [c for c in ['walkforward_scheme', 'window_label', 'retrain_frequency', 'test_year'] if c in predictions_df.columns]
    if include_fold:
        wf_cols += [c for c in ['fold_id', 'train_start_year', 'train_end_year', 'n_train_years'] if c in predictions_df.columns]
    return base_cols + wf_cols


def run_prediction_diagnostics(predictions_df, config=EXPERIMENT_CONFIG):
    if predictions_df is None or len(predictions_df) == 0:
        print('No predictions available.')
        return {}

    diagnostic_tables = {}
    group_cols = get_prediction_group_cols(predictions_df, include_fold=False)

    if config['run_daily_ranking']:
        parts = []
        for keys, g in predictions_df.groupby(group_cols, dropna=False):
            one = daily_cross_sectional_spearman(g)
            if len(one) == 0:
                continue
            for col, value in zip(group_cols, keys):
                one[col] = value
            parts.append(one)
        daily_df = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
        daily_summary = daily_df.groupby(group_cols, dropna=False).agg(
            n_days=('daily_spearman', 'count'),
            mean_daily_spearman=('daily_spearman', 'mean'),
            median_daily_spearman=('daily_spearman', 'median'),
            positive_spearman_day_rate=('daily_spearman', lambda x: (x > 0).mean()),
        ).reset_index() if len(daily_df) else pd.DataFrame()
        diagnostic_tables['daily_spearman_detail'] = register_table('daily_spearman_detail', daily_df)
        diagnostic_tables['daily_spearman_summary'] = register_table('daily_spearman_summary', daily_summary)

    if config['run_topk']:
        parts = []
        for top_pct in config['topk_percentages']:
            for keys, g in predictions_df.groupby(group_cols, dropna=False):
                one = topk_diagnostics(g, top_pct=top_pct)
                if len(one) == 0:
                    continue
                for col, value in zip(group_cols, keys):
                    one[col] = value
                parts.append(one)
        topk_df = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
        if len(topk_df):
            agg = {
                'n_days': ('top_minus_bottom_spread', 'count'),
                'mean_top_true': ('top_mean_true', 'mean'),
                'mean_bottom_true': ('bottom_mean_true', 'mean'),
                'mean_top_minus_bottom_spread': ('top_minus_bottom_spread', 'mean'),
                'positive_spread_day_rate': ('top_minus_bottom_spread', lambda x: (x > 0).mean()),
            }
            if 'precision_at_k' in topk_df.columns:
                agg['mean_precision_at_k'] = ('precision_at_k', 'mean')
            if 'lift_at_k' in topk_df.columns:
                agg['mean_lift_at_k'] = ('lift_at_k', 'mean')
            topk_summary = topk_df.groupby(group_cols + ['top_pct'], dropna=False).agg(**agg).reset_index()
        else:
            topk_summary = pd.DataFrame()
        diagnostic_tables['topk_detail'] = register_table('topk_detail', topk_df)
        diagnostic_tables['topk_summary'] = register_table('topk_summary', topk_summary)

    if config['run_within_ticker_ranking']:
        parts = []
        for keys, g in predictions_df.groupby(group_cols, dropna=False):
            one = within_ticker_ranking(g)
            if len(one) == 0:
                continue
            for col, value in zip(group_cols, keys):
                one[col] = value
            parts.append(one)
        w_df = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
        w_summary = w_df.groupby(group_cols, dropna=False).agg(
            n_tickers=('ticker', 'count'),
            mean_within_ticker_spearman=('within_ticker_spearman', 'mean'),
            median_within_ticker_spearman=('within_ticker_spearman', 'median'),
            positive_ticker_spearman_rate=('within_ticker_spearman', lambda x: (x > 0).mean()),
        ).reset_index() if len(w_df) else pd.DataFrame()
        diagnostic_tables['within_ticker_detail'] = register_table('within_ticker_detail', w_df)
        diagnostic_tables['within_ticker_summary'] = register_table('within_ticker_summary', w_summary)

    if config['run_subgroup_diagnostics']:
        for subgroup_col in [YEAR_COL, TICKER_COL, SIC2_COL]:
            if subgroup_col not in predictions_df.columns:
                continue
            parts = []
            for keys, g in predictions_df.groupby(group_cols, dropna=False):
                one = subgroup_prediction_metrics(g, subgroup_col)
                if len(one) == 0:
                    continue
                for col, value in zip(group_cols, keys):
                    one[col] = value
                parts.append(one)
            diagnostic_tables[f'subgroup_{subgroup_col}'] = register_table(
                f'subgroup_diagnostics_by_{subgroup_col}',
                pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame(),
            )

    if config['run_calibration']:
        parts = []
        binary_pred = predictions_df[predictions_df['task'] == 'binary'].copy()
        for keys, g in binary_pred.groupby(group_cols, dropna=False):
            one = calibration_table(g[TRUE_COL], g[SCORE_COL])
            if len(one) == 0:
                continue
            for col, value in zip(group_cols, keys):
                one[col] = value
            parts.append(one)
        diagnostic_tables['calibration'] = register_table(
            'binary_calibration_tables',
            pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame(),
        )

    return diagnostic_tables


def build_best_model_tables(metrics_df):
    if metrics_df is None or len(metrics_df) == 0:
        return {}
    df = metrics_df.copy()

    def choose_main_metric(row):
        if row['task'] == 'regression':
            return row.get('spearman', np.nan)
        if row['task'] == 'binary':
            return row.get('pr_auc', row.get('roc_auc', np.nan))
        if row['task'] == 'multiclass':
            return row.get('macro_f1', np.nan)
        return np.nan

    df['main_metric'] = df.apply(choose_main_metric, axis=1)

    grouping_context = [c for c in ['walkforward_scheme', 'window_label', 'retrain_frequency', 'test_year'] if c in df.columns]
    best_group_cols = ['target_name', SPLIT_COL] + grouping_context
    best_by_target = (
        df.dropna(subset=['main_metric'])
        .sort_values(best_group_cols + ['main_metric'], ascending=[True] * len(best_group_cols) + [False])
        .groupby(best_group_cols, as_index=False)
        .head(1)
        .reset_index(drop=True)
    ) if len(df.dropna(subset=['main_metric'])) else pd.DataFrame()

    feature_family_group_cols = ['target_name', 'task', 'feature_set', SPLIT_COL] + grouping_context
    feature_family = (
        df.dropna(subset=['main_metric'])
        .groupby(feature_family_group_cols, dropna=False)
        .agg(best_main_metric=('main_metric', 'max'), mean_main_metric=('main_metric', 'mean'), n_models=('model', 'nunique'))
        .reset_index()
        .sort_values(feature_family_group_cols + ['best_main_metric'], ascending=[True] * len(feature_family_group_cols) + [False])
    ) if len(df.dropna(subset=['main_metric'])) else pd.DataFrame()

    return {
        'best_by_target': register_table('best_model_by_target', best_by_target),
        'feature_family_comparison': register_table('feature_family_comparison', feature_family),
    }


def build_best_ranking_tables():
    tables = {}

    daily = EXPORTED_TABLES.get('daily_spearman_summary')
    if daily is not None and len(daily) > 0:
        context_cols = [c for c in ['walkforward_scheme', 'window_label', 'retrain_frequency', 'test_year'] if c in daily.columns]
        group_cols = ['target_name', SPLIT_COL] + context_cols
        best_daily = (
            daily.dropna(subset=['mean_daily_spearman'])
            .sort_values(group_cols + ['mean_daily_spearman'], ascending=[True] * len(group_cols) + [False])
            .groupby(group_cols, as_index=False)
            .head(1)
            .reset_index(drop=True)
        )
        tables['best_daily_spearman'] = register_table('best_daily_spearman_models', best_daily)

    topk = EXPORTED_TABLES.get('topk_summary')
    if topk is not None and len(topk) > 0:
        score_col = 'mean_lift_at_k' if 'mean_lift_at_k' in topk.columns else 'mean_top_minus_bottom_spread'
        context_cols = [c for c in ['walkforward_scheme', 'window_label', 'retrain_frequency', 'test_year'] if c in topk.columns]
        group_cols = ['target_name', SPLIT_COL, 'top_pct'] + context_cols
        best_topk = (
            topk.dropna(subset=[score_col])
            .sort_values(group_cols + [score_col], ascending=[True] * len(group_cols) + [False])
            .groupby(group_cols, as_index=False)
            .head(1)
            .reset_index(drop=True)
        )
        tables['best_topk'] = register_table('best_topk_models', best_topk)

    return tables

In [23]:
# ============================================================
# (A) Point-in-time scale-neutral cross-sectional Spearman & de-scaled Top-K
# ============================================================
# Wrap the score-percentile helper so it ALSO normalises the realised target per ticker
# against that ticker's TRAINING-window distribution (point-in-time, no look-ahead).
WITHIN_TICKER_TARGET_PCTILE_COL = 'within_ticker_target_percentile'
WITHIN_TICKER_TARGET_Z_COL = 'within_ticker_target_z'


def _pit_z_from_train(eval_vals, train_vals):
    tv = pd.to_numeric(pd.Series(train_vals), errors='coerce').dropna()
    ev = pd.to_numeric(pd.Series(eval_vals), errors='coerce')
    if len(tv) < 2 or tv.std(ddof=0) == 0:
        return pd.Series(np.nan, index=ev.index)
    return (ev - tv.mean()) / tv.std(ddof=0)


_orig_add_within_ticker_percentiles = add_historical_within_ticker_percentiles


def add_historical_within_ticker_percentiles(eval_pred_df, train_score_df, score_col=SCORE_COL):
    out = _orig_add_within_ticker_percentiles(eval_pred_df, train_score_df, score_col=score_col)
    if out is None or len(out) == 0 or TRUE_COL not in train_score_df.columns or TRUE_COL not in out.columns:
        return out
    out[WITHIN_TICKER_TARGET_PCTILE_COL] = np.nan
    out[WITHIN_TICKER_TARGET_Z_COL] = np.nan
    global_tpct = historical_percentile_from_train_distribution(out[TRUE_COL], train_score_df[TRUE_COL])
    if TICKER_COL in out.columns and TICKER_COL in train_score_df.columns:
        train_y = {t: g[TRUE_COL] for t, g in train_score_df.dropna(subset=[TRUE_COL]).groupby(TICKER_COL, dropna=False)}
        for ticker, idx in out.groupby(TICKER_COL, dropna=False).groups.items():
            idx = list(idx)
            if ticker in train_y and pd.Series(train_y[ticker]).dropna().shape[0] > 1:
                out.loc[idx, WITHIN_TICKER_TARGET_PCTILE_COL] = historical_percentile_from_train_distribution(out.loc[idx, TRUE_COL], train_y[ticker]).values
                out.loc[idx, WITHIN_TICKER_TARGET_Z_COL] = _pit_z_from_train(out.loc[idx, TRUE_COL], train_y[ticker]).values
            else:
                out.loc[idx, WITHIN_TICKER_TARGET_PCTILE_COL] = global_tpct.loc[idx].values
    else:
        out[WITHIN_TICKER_TARGET_PCTILE_COL] = global_tpct.values
    return out


def _sn_group_cols(df):
    base = ['target_name', 'task', 'feature_set', 'model', SPLIT_COL]
    extra = [c for c in ['walkforward_scheme', 'window_label', 'test_year'] if c in df.columns]
    return base + extra


def scale_neutral_cross_sectional_diagnostics(predictions_df, min_daily_rows=5):
    if predictions_df is None or len(predictions_df) == 0 or WITHIN_TICKER_TARGET_PCTILE_COL not in predictions_df.columns:
        print('Scale-neutral diagnostics skipped: within-ticker target percentile not present.')
        return {}
    group_cols = _sn_group_cols(predictions_df)
    parts = []
    for keys, g in predictions_df.groupby(group_cols, dropna=False):
        if DATE_COL not in g.columns:
            continue
        rows = []
        for date, gd in g.groupby(DATE_COL):
            gd = gd.dropna(subset=[WITHIN_TICKER_PERCENTILE_SCORE_COL, WITHIN_TICKER_TARGET_PCTILE_COL])
            if len(gd) < min_daily_rows:
                continue
            rows.append({DATE_COL: date, 'n': len(gd),
                         'raw_daily_spearman': safe_spearman(gd[SCORE_COL], gd[TRUE_COL]),
                         'scaleneutral_daily_spearman': safe_spearman(gd[WITHIN_TICKER_PERCENTILE_SCORE_COL], gd[WITHIN_TICKER_TARGET_PCTILE_COL])})
        one = pd.DataFrame(rows)
        if len(one) == 0:
            continue
        for c, v in zip(group_cols, keys):
            one[c] = v
        parts.append(one)
    detail = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
    if len(detail):
        summary = detail.groupby(group_cols, dropna=False).agg(
            n_days=('scaleneutral_daily_spearman', 'count'),
            mean_raw_daily_spearman=('raw_daily_spearman', 'mean'),
            mean_scaleneutral_daily_spearman=('scaleneutral_daily_spearman', 'mean'),
            median_scaleneutral_daily_spearman=('scaleneutral_daily_spearman', 'median'),
            positive_day_rate=('scaleneutral_daily_spearman', lambda x: (x > 0).mean())).reset_index()
    else:
        summary = pd.DataFrame()
    register_table('scaleneutral_cross_sectional_detail', detail)
    register_table('scaleneutral_cross_sectional_summary', summary)
    return {'detail': detail, 'summary': summary}


def descaled_topk_diagnostics(predictions_df, top_pcts=(0.05, 0.10), min_daily_rows=10):
    if predictions_df is None or len(predictions_df) == 0 or WITHIN_TICKER_TARGET_PCTILE_COL not in predictions_df.columns:
        print('De-scaled Top-K skipped: within-ticker target percentile not present.')
        return {}
    group_cols = _sn_group_cols(predictions_df)
    sel = WITHIN_TICKER_PERCENTILE_SCORE_COL
    parts = []
    for keys, g in predictions_df.groupby(group_cols, dropna=False):
        if DATE_COL not in g.columns:
            continue
        for tp in top_pcts:
            day_rows = []; picks = []
            for date, gd in g.groupby(DATE_COL):
                gd = gd.dropna(subset=[sel, WITHIN_TICKER_TARGET_PCTILE_COL])
                if len(gd) < min_daily_rows:
                    continue
                k = max(1, int(np.ceil(len(gd) * tp)))
                top = gd.nlargest(k, sel); bot = gd.nsmallest(k, sel)
                sp_z = (top[WITHIN_TICKER_TARGET_Z_COL].mean() - bot[WITHIN_TICKER_TARGET_Z_COL].mean()) if WITHIN_TICKER_TARGET_Z_COL in gd.columns else np.nan
                raw_top = pd.to_numeric(top[TRUE_COL], errors='coerce').mean()
                raw_bot = pd.to_numeric(bot[TRUE_COL], errors='coerce').mean()
                day_rows.append({DATE_COL: date, 'n': len(gd), 'k': k, 'top_pct': tp,
                                 'spread_target_percentile': top[WITHIN_TICKER_TARGET_PCTILE_COL].mean() - bot[WITHIN_TICKER_TARGET_PCTILE_COL].mean(),
                                 'spread_target_z': sp_z,
                                 'top_raw_target': raw_top, 'bot_raw_target': raw_bot,
                                 'spread_target_raw': raw_top - raw_bot})
                if TICKER_COL in top.columns:
                    picks.extend(top[TICKER_COL].tolist())
            one = pd.DataFrame(day_rows)
            if len(one) == 0:
                continue
            picks = pd.Series(picks)
            one['unique_tickers_in_topk'] = picks.nunique()
            one['top_name_share'] = picks.value_counts(normalize=True).iloc[0] if len(picks) else np.nan
            for c, v in zip(group_cols, keys):
                one[c] = v
            parts.append(one)
    detail = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
    if len(detail):
        summary = detail.groupby(group_cols + ['top_pct'], dropna=False).agg(
            n_days=('spread_target_percentile', 'count'),
            mean_spread_target_percentile=('spread_target_percentile', 'mean'),
            mean_spread_target_z=('spread_target_z', 'mean'),
            mean_top_raw_target=('top_raw_target', 'mean'),
            mean_bot_raw_target=('bot_raw_target', 'mean'),
            mean_spread_target_raw=('spread_target_raw', 'mean'),
            positive_spread_day_rate=('spread_target_percentile', lambda x: (x > 0).mean()),
            positive_raw_spread_day_rate=('spread_target_raw', lambda x: (x > 0).mean()),
            unique_tickers_in_topk=('unique_tickers_in_topk', 'max'),
            top_name_share=('top_name_share', 'max')).reset_index()
    else:
        summary = pd.DataFrame()
    register_table('descaled_topk_detail', detail)
    register_table('descaled_topk_summary', summary)
    return {'detail': detail, 'summary': summary}

def scale_neutral_decile_diagnostics(predictions_df, n_deciles=10, min_daily_rows=10):
    """Per-day cross-sectional deciles by the scale-neutral (within-ticker) score percentile.
    On each trading day the stocks are ranked into n_deciles by within_ticker_score_percentile;
    per decile the mean realised within-ticker target percentile and positive-outcome rate are
    formed for that day, then averaged over trading days (reported per fold-group). Reproduces
    the scale-neutral decile figure. mean_realised_raw is included for reference only."""
    sel = WITHIN_TICKER_PERCENTILE_SCORE_COL
    if (predictions_df is None or len(predictions_df) == 0
            or WITHIN_TICKER_TARGET_PCTILE_COL not in predictions_df.columns or sel not in predictions_df.columns):
        print('Scale-neutral decile skipped: within-ticker percentiles not present.')
        return {}
    group_cols = _sn_group_cols(predictions_df)
    parts = []
    for keys, g in predictions_df.groupby(group_cols, dropna=False):
        if DATE_COL not in g.columns:
            continue
        day_rows = []
        for date, gd in g.groupby(DATE_COL):
            gd = gd.dropna(subset=[sel, WITHIN_TICKER_TARGET_PCTILE_COL])
            if len(gd) < min_daily_rows:
                continue
            r = gd[sel].rank(method='first', pct=True)
            dec = np.ceil(r * n_deciles).clip(1, n_deciles).astype(int)
            tmp = pd.DataFrame({'decile': dec.values,
                                'tp': gd[WITHIN_TICKER_TARGET_PCTILE_COL].values,
                                'pos': (pd.to_numeric(gd[TRUE_COL], errors='coerce') > 0).astype(float).values,
                                'raw': pd.to_numeric(gd[TRUE_COL], errors='coerce').values})
            agg = tmp.groupby('decile').agg(mean_realised_pct=('tp', 'mean'),
                                            positive_rate=('pos', 'mean'),
                                            mean_realised_raw=('raw', 'mean'),
                                            n=('tp', 'size')).reset_index()
            agg[DATE_COL] = date
            day_rows.append(agg)
        if not day_rows:
            continue
        alld = pd.concat(day_rows, ignore_index=True)
        summ = alld.groupby('decile').agg(n_days=('n', 'size'),
                                          mean_realised_pct=('mean_realised_pct', 'mean'),
                                          positive_rate=('positive_rate', 'mean'),
                                          mean_realised_raw=('mean_realised_raw', 'mean')).reset_index()
        for c, v in zip(group_cols, keys if isinstance(keys, tuple) else (keys,)):
            summ[c] = v
        parts.append(summ)
    out = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
    register_table('scaleneutral_cross_sectional_decile', out)
    return {'summary': out}



In [24]:
# ============================================================
# (B) Per-year (per-fold) within-ticker skill: spread (raw / z / percentile) & decile gradient
# ============================================================
# Normalisation is per (fold x ticker): each ticker's realised target is expressed as a
# z-score and a percentile within that fold's days (operation 1 = scale/amplitude-neutral),
# and days are ranked into within-ticker deciles by predicted score. Fold = walk-forward
# test year (or, in the fixed split, split x calendar year) => operation 2 = per-year.
# Pooled tables average over full folds only (partial years auto-excluded; the parquet keeps
# every fold, so a full-including-partial recompute is always possible afterwards).

def _within_ticker_fold_cols(df):
    base = ['target_name', 'feature_set', 'model']
    if 'walkforward_scheme' in df.columns and 'test_year' in df.columns:
        return base + ['walkforward_scheme', 'test_year'], 'test_year'
    yk = YEAR_COL if (YEAR_COL in df.columns) else None
    return base + [c for c in [SPLIT_COL, yk] if c is not None], yk


def _partial_year_set(df, year_key, ratio=0.75):
    if year_key is None or DATE_COL not in df.columns:
        return set()
    nd = df.dropna(subset=[year_key]).groupby(year_key)[DATE_COL].nunique()
    if len(nd) == 0:
        return set()
    return set(nd[nd < ratio * nd.median()].index)


def _augment_within_ticker_fold(df, n_deciles=10, min_days_per_ticker=20):
    keys, year_key = _within_ticker_fold_cols(df)
    g = df.dropna(subset=[SCORE_COL, TRUE_COL, TICKER_COL]).copy()
    g[TRUE_COL] = pd.to_numeric(g[TRUE_COL], errors='coerce')
    g[SCORE_COL] = pd.to_numeric(g[SCORE_COL], errors='coerce')
    g = g.dropna(subset=[TRUE_COL, SCORE_COL])
    grp = keys + [TICKER_COL]
    g = g[g.groupby(grp)[SCORE_COL].transform('size') >= min_days_per_ticker].copy()
    if len(g) == 0:
        return g, keys, year_key
    g['_pct'] = g.groupby(grp)[TRUE_COL].rank(pct=True)
    g['_z'] = g.groupby(grp)[TRUE_COL].transform(lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) > 0 else np.nan)

    def _dec(s):
        r = s.rank(method='first')
        return np.ceil(r / len(s) * n_deciles).clip(1, n_deciles).astype(int)

    g['_decile'] = g.groupby(grp)[SCORE_COL].transform(_dec)
    return g, keys, year_key


def within_ticker_spread_by_fold(predictions_df, top_pcts=(0.05, 0.10), min_days_per_ticker=20, min_tickers=5):
    if predictions_df is None or len(predictions_df) == 0:
        return {}
    g, keys, year_key = _augment_within_ticker_fold(predictions_df, min_days_per_ticker=min_days_per_ticker)
    if len(g) == 0:
        print('within_ticker_spread_by_fold: no eligible rows.')
        return {}
    rows = []
    for fold_vals, gf in g.groupby(keys, dropna=False):
        fv = fold_vals if isinstance(fold_vals, tuple) else (fold_vals,)
        for tp in top_pcts:
            sp_raw = []; sp_z = []; sp_pct = []
            for tk, gt in gf.groupby(TICKER_COL):
                k = max(1, int(np.ceil(len(gt) * tp)))
                order = gt[SCORE_COL].rank(ascending=False, method='first')
                top = order <= k; bot = order > len(gt) - k
                sp_raw.append(gt[TRUE_COL][top].mean() - gt[TRUE_COL][bot].mean())
                sp_z.append(gt['_z'][top].mean() - gt['_z'][bot].mean())
                sp_pct.append(gt['_pct'][top].mean() - gt['_pct'][bot].mean())
            if len(sp_raw) < min_tickers:
                continue
            row = dict(zip(keys, fv))
            row.update(top_pct=tp, n_tickers=len(sp_raw),
                       mean_spread_raw=np.nanmean(sp_raw),
                       mean_spread_z=np.nanmean(sp_z),
                       mean_spread_pct=np.nanmean(sp_pct),
                       positive_spread_ticker_rate=np.nanmean([1.0 if x > 0 else 0.0 for x in sp_raw]))
            rows.append(row)
    detail = pd.DataFrame(rows)
    register_table('within_ticker_spread_by_fold', detail)
    pooled = pd.DataFrame()
    if len(detail) and year_key in detail.columns:
        partial = _partial_year_set(predictions_df, year_key)
        det = detail[~detail[year_key].isin(partial)]
        pool_keys = [k for k in keys if k != year_key] + ['top_pct']
        if len(det):
            pooled = det.groupby(pool_keys, dropna=False).agg(
                n_folds=('mean_spread_pct', 'count'),
                mean_spread_raw=('mean_spread_raw', 'mean'),
                mean_spread_z=('mean_spread_z', 'mean'),
                mean_spread_pct=('mean_spread_pct', 'mean'),
                mean_positive_spread_ticker_rate=('positive_spread_ticker_rate', 'mean')).reset_index()
            pooled['partial_years_excluded'] = (','.join(map(str, sorted(partial))) if partial else 'none')
    register_table('within_ticker_spread_pooled', pooled)
    return {'by_fold': detail, 'pooled': pooled}


def within_ticker_decile_diagnostics(predictions_df, n_deciles=10, min_days_per_ticker=20):
    if predictions_df is None or len(predictions_df) == 0:
        return {}
    g, keys, year_key = _augment_within_ticker_fold(predictions_df, n_deciles=n_deciles, min_days_per_ticker=min_days_per_ticker)
    if len(g) == 0:
        print('within_ticker_decile_diagnostics: no eligible rows.')
        return {}

    def _agg(frame, group_cols):
        return frame.groupby(group_cols + ['_decile'], dropna=False).agg(
            mean_realised_raw=(TRUE_COL, 'mean'),
            mean_realised_z=('_z', 'mean'),
            mean_realised_pct=('_pct', 'mean'),
            positive_rate=(TRUE_COL, lambda x: (pd.to_numeric(x, errors='coerce') > 0).mean()),
            n=(TRUE_COL, 'size')).reset_index().rename(columns={'_decile': 'decile'})

    by_fold = _agg(g, keys)
    register_table('within_ticker_decile_by_fold', by_fold)
    partial = _partial_year_set(predictions_df, year_key) if year_key else set()
    gg = g[~g[year_key].isin(partial)] if year_key else g
    pooled = _agg(gg, [k for k in keys if k != year_key])
    if year_key:
        pooled['partial_years_excluded'] = (','.join(map(str, sorted(partial))) if partial else 'none')
    register_table('within_ticker_decile_pooled', pooled)
    return {'by_fold': by_fold, 'pooled': pooled}


In [ ]:
# ============================================================
# (C) Ticker heterogeneity, SIC2 sector aggregation, dependence-robust CIs,
#     signed-log error metrics, and the within-ticker decile figure
# ============================================================

def _pooled_per_ticker_within_spearman(predictions_df):
    keys, year_key = _within_ticker_fold_cols(predictions_df)
    keys_noyear = [k for k in keys if k != year_key]
    df = predictions_df.dropna(subset=[SCORE_COL, TRUE_COL, TICKER_COL]).copy()
    if year_key is not None:
        df = df[~df[year_key].isin(_partial_year_set(predictions_df, year_key))]
    rows = []
    for gv, g in df.groupby(keys_noyear + [TICKER_COL], dropna=False):
        s = safe_spearman(g[SCORE_COL], g[TRUE_COL])
        if s != s:
            continue
        row = dict(zip(keys_noyear + ['ticker'], gv if isinstance(gv, tuple) else (gv,)))
        row['within_ticker_spearman'] = s
        row['n'] = len(g)
        row['sic2'] = g[SIC2_COL].iloc[0] if SIC2_COL in g.columns else np.nan
        rows.append(row)
    return pd.DataFrame(rows), keys_noyear


def _load_gics_map():
    """Map ticker -> {gics_industry, gics_subsector} from the ticker-universe workbook.
    Searches the working dir (and DATA_DIR/INPUT_DIR/BASE_DIR if defined) for an .xlsx whose
    name mentions 'universe' or 'ticker...summary'. Returns {} if none found, so the rest of
    the diagnostics still run on SIC2 only."""
    import glob as _glob, os as _os
    search_dirs = ['.']
    for v in ['DATA_DIR', 'INPUT_DIR', 'BASE_DIR']:
        p = globals().get(v)
        if p is not None:
            search_dirs.append(str(p))
    cands = []
    for d in search_dirs:
        for pat in ['*[Uu]niverse*.xlsx', '*[Tt]icker*[Ss]ummary*.xlsx']:
            cands += _glob.glob(_os.path.join(d, pat))
    cands = list(dict.fromkeys(cands))
    if not cands:
        print('GICS map: no ticker-universe workbook found (sector heterogeneity uses SIC2 only).')
        return {}
    path = cands[0]
    m = {}
    try:
        xl = pd.ExcelFile(path)
        for sh in xl.sheet_names:
            df = xl.parse(sh)
            low = {str(c).strip().lower(): c for c in df.columns}
            tcol = next((low[k] for k in ('ticker', 'symbol') if k in low), None)
            icol = low.get('industry'); scol = low.get('subsector')
            if tcol is None:
                continue
            for _, r in df.iterrows():
                t = str(r[tcol]).strip().upper()
                if not t or t == 'NAN':
                    continue
                m[t] = {'gics_industry': (str(r[icol]).strip() if icol is not None and pd.notna(r[icol]) else np.nan),
                        'gics_subsector': (str(r[scol]).strip() if scol is not None and pd.notna(r[scol]) else np.nan)}
    except Exception as e:
        print('GICS map load failed:', e)
        return {}
    print(f'GICS map: {len(m)} tickers loaded from {path}')
    return m


def ticker_heterogeneity_diagnostics(predictions_df):
    detail, keys_noyear = _foldavg_per_ticker_within_spearman(predictions_df)
    if len(detail) == 0:
        print('ticker heterogeneity: no eligible rows.')
        return {}
    # Attach GICS Industry / Subsector from the ticker-universe workbook (if present).
    gmap = _load_gics_map()
    if gmap:
        up = detail['ticker'].astype(str).str.upper()
        detail['gics_industry'] = up.map(lambda t: gmap.get(t, {}).get('gics_industry', np.nan))
        detail['gics_subsector'] = up.map(lambda t: gmap.get(t, {}).get('gics_subsector', np.nan))
    register_table('ticker_heterogeneity_detail', detail)
    summ = detail.groupby(keys_noyear, dropna=False)['within_ticker_spearman'].agg(
        n_tickers='count', mean='mean', median='median', std='std',
        q25=lambda x: x.quantile(0.25), q75=lambda x: x.quantile(0.75),
        min='min', max='max', positive_ticker_rate=lambda x: (x > 0).mean()).reset_index()
    register_table('ticker_heterogeneity_summary', summ)
    if 'sic2' in detail.columns:
        sec = detail.groupby(keys_noyear + ['sic2'], dropna=False)['within_ticker_spearman'].agg(
            n_tickers='count', mean='mean', median='median', std='std').reset_index()
        register_table('ticker_heterogeneity_by_sic2', sec)
    # GICS sector aggregation (Industry = broad, Subsector = fine).
    for gcol, tname in [('gics_industry', 'ticker_heterogeneity_by_gics_industry'),
                        ('gics_subsector', 'ticker_heterogeneity_by_gics_subsector')]:
        if gcol in detail.columns and detail[gcol].notna().any():
            sec = detail.dropna(subset=[gcol]).groupby(keys_noyear + [gcol], dropna=False)['within_ticker_spearman'].agg(
                n_tickers='count', mean='mean', median='median', std='std').reset_index()
            register_table(tname, sec)
    # Cross-model consistency of the per-ticker within-ticker Spearman (e.g. RF vs LGBM: rho, %same-sign).
    grp0 = [k for k in keys_noyear if k != 'model']
    cm = []
    for gv, g in (detail.groupby(grp0, dropna=False) if grp0 else [((), detail)]):
        base = dict(zip(grp0, gv if isinstance(gv, tuple) else (gv,)))
        for a, b in (('RandomForest', 'LightGBM'), ('RandomForest', 'ElasticNet'), ('LightGBM', 'ElasticNet')):
            pa = g[g['model'] == a].set_index('ticker')['within_ticker_spearman']
            pb = g[g['model'] == b].set_index('ticker')['within_ticker_spearman']
            ix = pa.index.intersection(pb.index)
            if len(ix) < 3:
                continue
            xa = pa[ix].values; xb = pb[ix].values
            cm.append({**base, 'model_pair': f'{a} vs {b}', 'n_tickers': int(len(ix)),
                       'spearman_rho': safe_spearman(xa, xb),
                       'pearson_r': float(np.corrcoef(xa, xb)[0, 1]),
                       'pct_same_sign': float(np.mean(np.sign(xa) == np.sign(xb))),
                       'pct_both_positive': float(np.mean((xa > 0) & (xb > 0)))})
    if cm:
        register_table('cross_model_within_ticker_consistency', pd.DataFrame(cm))
    return {'detail': detail, 'summary': summ}

# ------------------------------------------------------------
# Alternative CI diagnostics (not used for final reported within-ticker CIs)
# ------------------------------------------------------------
# Retained for sensitivity checks and comparison of interval widths under
# alternative resampling assumptions. Final dissertation within-ticker CIs
# are computed in Section 12.3 using the fold-stratified 21-day moving-block bootstrap.

def _block_boot_mean(series, B=1000, blk=21, seed=0):
    rng = np.random.default_rng(seed)
    s = np.asarray([x for x in series if x == x], dtype=float); n = len(s)
    if n < 3:
        return (np.nan, np.nan, np.nan)
    nb = int(np.ceil(n / blk)); out = []
    for _ in range(B):
        st = rng.integers(0, n, nb)
        idx = np.concatenate([np.arange(x, x + blk) % n for x in st])[:n]
        out.append(s[idx].mean())
    return (float(np.nanmean(s)), float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5)))


def _boot_mean(vals, B=1000, seed=0):
    rng = np.random.default_rng(seed)
    s = np.asarray([x for x in vals if x == x], dtype=float); n = len(s)
    if n < 3:
        return (np.nan, np.nan, np.nan)
    out = [s[rng.integers(0, n, n)].mean() for _ in range(B)]
    return (float(s.mean()), float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5)))


def _fold_block_boot_mean(fold_series, B=1000, blk=21, seed=0):
    """Fold-stratified moving-block bootstrap of a fold-averaged daily-correlation mean.
    Resamples 21-day blocks WITHIN each fold (non-circular), takes each fold mean, then
    averages across folds (Fama-MacBeth). Returns (point, ci_low, ci_high)."""
    rng = np.random.default_rng(seed)
    folds = [np.asarray([x for x in ser if x == x], dtype=float) for ser in fold_series]
    folds = [ser for ser in folds if len(ser) >= 3]
    if not folds:
        return (np.nan, np.nan, np.nan)
    point = float(np.mean([ser.mean() for ser in folds]))
    out = []
    for _ in range(B):
        fm = []
        for ser in folds:
            n = len(ser); eff = min(blk, n); nbk = int(np.ceil(n / eff))
            starts = rng.integers(0, n - eff + 1, nbk)
            idx = np.concatenate([np.arange(st, st + eff) for st in starts])[:n]
            fm.append(ser[idx].mean())
        out.append(float(np.mean(fm)))
    return (point, float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5)))


def ranking_confidence_intervals(predictions_df, B=1000, block=21,
                                 paired_models=(('LightGBM', 'ElasticNet'),
                                                ('RandomForest', 'ElasticNet'),
                                                ('RandomForest', 'LightGBM'))):
    keys, year_key = _within_ticker_fold_cols(predictions_df)
    grp = [k for k in keys if k != 'model' and k != year_key]
    df = predictions_df.copy()
    if year_key is not None:
        df = df[~df[year_key].isin(_partial_year_set(predictions_df, year_key))]
    rows = []; paired_rows = []
    for gv, g in df.groupby(grp, dropna=False):
        base = dict(zip(grp, gv if isinstance(gv, tuple) else (gv,)))
        wt = {}
        for mo, gm in g.groupby('model'):
            per = {}
            for tk, x in gm.groupby(TICKER_COL):
                s = safe_spearman(x[SCORE_COL], x[TRUE_COL])
                if s == s:
                    per[tk] = s
            per = pd.Series(per)
            wt[mo] = per
            p, lo, hi = _boot_mean(per.values, B=B)
            rows.append({**base, 'model': mo, 'metric': 'within_ticker_spearman',
                         'point': p, 'ci_low': lo, 'ci_high': hi, 'n': int(per.notna().sum())})
        if WITHIN_TICKER_TARGET_PCTILE_COL in g.columns and WITHIN_TICKER_PERCENTILE_SCORE_COL in g.columns:
            for mo, gm in g.groupby('model'):
                fold_series = []
                for _fy, gy in (gm.groupby(year_key) if year_key else [(None, gm)]):
                    ds = []
                    for _, gd in gy.groupby(DATE_COL):
                        gd = gd.dropna(subset=[WITHIN_TICKER_PERCENTILE_SCORE_COL, WITHIN_TICKER_TARGET_PCTILE_COL])
                        if len(gd) >= 5:
                            ds.append(safe_spearman(gd[WITHIN_TICKER_PERCENTILE_SCORE_COL], gd[WITHIN_TICKER_TARGET_PCTILE_COL]))
                    if ds:
                        fold_series.append(ds)
                p, lo, hi = _fold_block_boot_mean(fold_series, B=B, blk=block)
                rows.append({**base, 'model': mo, 'metric': 'scaleneutral_cross_sectional_spearman',
                             'point': p, 'ci_low': lo, 'ci_high': hi,
                             'n': int(sum(len(sr) for sr in fold_series))})
        for a, b in paired_models:
            if a in wt and b in wt:
                idx = wt[a].index.intersection(wt[b].index)
                dif = (wt[a][idx] - wt[b][idx]).values
                p, lo, hi = _boot_mean(dif, B=B)
                paired_rows.append({**base, 'contrast': f'{a} - {b}', 'metric': 'within_ticker_spearman',
                                    'diff': p, 'ci_low': lo, 'ci_high': hi, 'reliable': bool(lo > 0 or hi < 0)})
    register_table('ranking_ci', pd.DataFrame(rows))
    register_table('ranking_ci_paired', pd.DataFrame(paired_rows))
    return {'ci': pd.DataFrame(rows), 'paired': pd.DataFrame(paired_rows)}


def signedlog_error_metrics(predictions_df):
    """R2 / RMSE / MAE computed on the signed-log mapping of (y_true, y_pred) from the
    RAW-target model, per fold and model -- the 'one headline model, error on signed-log' version."""
    keys, year_key = _within_ticker_fold_cols(predictions_df)
    g = predictions_df.dropna(subset=[TRUE_COL, PRED_COL]).copy()

    def sl(x):
        x = pd.to_numeric(x, errors='coerce')
        return np.sign(x) * np.log1p(np.abs(x))

    g['_slt'] = sl(g[TRUE_COL]); g['_slp'] = sl(g[PRED_COL])
    rows = []
    for gv, gg in g.groupby(keys, dropna=False):
        yt = gg['_slt']; yp = gg['_slp']; m = yt.notna() & yp.notna()
        if m.sum() < 3:
            continue
        yt = yt[m]; yp = yp[m]; ss = ((yt - yt.mean()) ** 2).sum()
        r2 = 1 - ((yt - yp) ** 2).sum() / ss if ss > 0 else np.nan
        row = dict(zip(keys, gv if isinstance(gv, tuple) else (gv,)))
        row.update(r2_signedlog=r2, rmse_signedlog=float(np.sqrt(((yt - yp) ** 2).mean())),
                   mae_signedlog=float((yt - yp).abs().mean()), n=int(m.sum()))
        rows.append(row)
    t = pd.DataFrame(rows)
    register_table('error_metrics_signedlog', t)
    return t


def plot_within_ticker_decile(predictions_df=None):
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
    except Exception as e:
        print('decile plot skipped:', e)
        return
    dp = EXPORTED_TABLES.get('within_ticker_decile_pooled')
    if dp is None or len(dp) == 0:
        print('decile plot: no within_ticker_decile_pooled table.')
        return
    keys = [c for c in ['target_name', 'walkforward_scheme'] if c in dp.columns]
    col = {'RandomForest': '#1f4e79', 'LightGBM': '#2e9e4f', 'ElasticNet': '#e07b28'}
    for gv, g in (dp.groupby(keys) if keys else [((), dp)]):
        fig, ax = plt.subplots(1, 2, figsize=(11, 4.2), dpi=300)
        for mo, gm in g.groupby('model'):
            gm = gm.sort_values('decile')
            ax[0].plot(gm['decile'], gm['mean_realised_pct'], marker='o', label=mo, color=col.get(mo))
            ax[1].plot(gm['decile'], gm['positive_rate'], marker='o', label=mo, color=col.get(mo))
        ax[0].axhline(0.5, ls='--', c='0.5'); ax[1].axhline(0.5, ls='--', c='0.5')
        ax[0].set_title('(a) mean realised within-ticker percentile'); ax[1].set_title('(b) positive-outcome rate')
        for a in ax:
            a.set_xlabel('within-ticker prediction decile'); a.legend(fontsize=8); a.grid(alpha=0.2)
        tag = safe_file_name('_'.join(str(x) for x in (gv if isinstance(gv, tuple) else (gv,)))) or 'decile'
        path = FIGURE_DIR / f'within_ticker_decile_{tag}.png'
        plt.tight_layout(); plt.savefig(path, bbox_inches='tight'); plt.close()
        print('saved figure:', path)


# === Reconciliation additions (fold-average heterogeneity, within-ticker ROC-AUC, PnL-scale Pearson) ===
def _foldavg_per_ticker_within_spearman(predictions_df):
    """Per-ticker within-ticker Spearman computed PER FOLD, then averaged over
    folds per ticker (partial years excluded). Replaces the pooled version."""
    keys, year_key = _within_ticker_fold_cols(predictions_df)
    keys_noyear = [k for k in keys if k != year_key]
    df = predictions_df.dropna(subset=[SCORE_COL, TRUE_COL, TICKER_COL]).copy()
    if year_key is not None:
        df = df[~df[year_key].isin(_partial_year_set(predictions_df, year_key))]
    grp = keys_noyear + [TICKER_COL] + ([year_key] if year_key else [])
    per_fold = []
    for gv, g in df.groupby(grp, dropna=False):
        s = safe_spearman(g[SCORE_COL], g[TRUE_COL])
        if s != s:
            continue
        row = dict(zip(grp, gv if isinstance(gv, tuple) else (gv,)))
        row['_rho'] = s; row['_n'] = len(g)
        per_fold.append(row)
    pf = pd.DataFrame(per_fold)
    if len(pf) == 0:
        return pd.DataFrame(), keys_noyear
    agg = (pf.groupby(keys_noyear + [TICKER_COL], dropna=False)
             .agg(within_ticker_spearman=('_rho', 'mean'),
                  n_folds=('_rho', 'size'), n=('_n', 'sum')).reset_index()
             .rename(columns={TICKER_COL: 'ticker'}))
    if SIC2_COL in df.columns:
        sic = (df.groupby(keys_noyear + [TICKER_COL])[SIC2_COL].first().reset_index()
                 .rename(columns={TICKER_COL: 'ticker', SIC2_COL: 'sic2'}))
        agg = agg.merge(sic, on=keys_noyear + ['ticker'], how='left')
    return agg, keys_noyear


def within_ticker_rocauc(predictions_df, targets=None, min_ticker_rows=20):
    """Within-ticker ROC-AUC for binary targets: per-ticker AUC per fold ->
    fold-average per ticker -> mean over tickers (+ fold range)."""
    from sklearn.metrics import roc_auc_score
    if targets is None:
        targets = BINARY_TARGETS
    keys, year_key = _within_ticker_fold_cols(predictions_df)
    keys_noyear = [k for k in keys if k != year_key]
    df = predictions_df.dropna(subset=[SCORE_COL, TRUE_COL, TICKER_COL]).copy()
    df = df[df['target_name'].isin(targets)]
    if year_key is not None:
        df = df[~df[year_key].isin(_partial_year_set(predictions_df, year_key))]
    grp = keys_noyear + [TICKER_COL] + ([year_key] if year_key else [])
    per = []
    for gv, g in df.groupby(grp, dropna=False):
        y = pd.to_numeric(g[TRUE_COL], errors='coerce')
        s = pd.to_numeric(g[SCORE_COL], errors='coerce')
        m = y.notna() & s.notna()
        if m.sum() < min_ticker_rows or y[m].nunique() < 2:
            continue
        row = dict(zip(grp, gv if isinstance(gv, tuple) else (gv,)))
        row['_auc'] = roc_auc_score(y[m], s[m]); per.append(row)
    pf = pd.DataFrame(per)
    if len(pf) == 0:
        return pf
    tick = pf.groupby(keys_noyear + [TICKER_COL]).agg(auc=('_auc', 'mean')).reset_index()
    out = (tick.groupby(keys_noyear)
              .agg(within_ticker_roc_auc=('auc', 'mean'), n_tickers=('auc', 'size')).reset_index())
    if year_key:
        perfold = pf.groupby(keys_noyear + [year_key]).agg(m=('_auc', 'mean')).reset_index()
        rng = perfold.groupby(keys_noyear).agg(fold_min=('m', 'min'), fold_max=('m', 'max')).reset_index()
        out = out.merge(rng, on=keys_noyear, how='left')
    register_table('within_ticker_rocauc', out)
    return out


def _pnl_scale_initial_window(pnl_target='target__rl_long_current_pnl', n_years=None):
    """Per-ticker PnL scale = std of realised current PnL measured over the INITIAL training
    window only -- the earliest `n_years` calendar years present in the sample. That window
    precedes every walk-forward test fold, so the scale is point-in-time (no look-ahead) and
    is measured on one common basis for all tickers. `n_years` defaults to the rolling-window
    length (2), so the window is exactly the first rolling_2y / first expanding training fold.
    Returns DataFrame[ticker, pnl_std] or None if the target/columns are unavailable."""
    src = None
    if 'all_data' in globals() and pnl_target in getattr(all_data, 'columns', []):
        src = all_data
    elif 'frames' in globals() and pnl_target in frames['train'].columns:
        src = frames['train']
    if src is None or TICKER_COL not in src.columns:
        return None
    if YEAR_COL in src.columns:
        yr = pd.to_numeric(src[YEAR_COL], errors='coerce')
    elif DATE_COL in src.columns:
        yr = pd.to_datetime(src[DATE_COL], errors='coerce').dt.year
    else:
        return None
    if n_years is None:
        n_years = int(WALK_FORWARD_CONFIG.get('rolling_window_years', 2)) if 'WALK_FORWARD_CONFIG' in globals() else 2
    years_sorted = sorted(pd.Series(yr).dropna().astype(int).unique().tolist())
    if not years_sorted:
        return None
    window_years = years_sorted[:n_years]
    m = yr.isin(window_years) & src[pnl_target].notna() & src[TICKER_COL].notna()
    sc = (src.loc[m].groupby(TICKER_COL)[pnl_target].std()
             .rename('pnl_std').reset_index().rename(columns={TICKER_COL: 'ticker'}))
    print(f'PnL scale: std over initial training window {window_years} '
          f'({sc["pnl_std"].notna().sum()} tickers, point-in-time, no look-ahead).')
    return sc


def within_ranking_vs_pnl_scale(predictions_df, detail=None, pnl_target='target__rl_long_current_pnl'):
    """Pearson between per-ticker within-Spearman (fold-average) and log(PnL scale).
    PnL scale = std of realised current PnL over the INITIAL training window
    (earliest 2 sample years; precedes every test fold -> point-in-time, no look-ahead)."""
    if detail is None or 'within_ticker_spearman' not in getattr(detail, 'columns', []):
        detail, keys_noyear = _foldavg_per_ticker_within_spearman(predictions_df)
    else:
        keys_noyear = [c for c in ['target_name', 'feature_set', 'model', 'walkforward_scheme']
                       if c in detail.columns]
    sc = _pnl_scale_initial_window(pnl_target)
    if sc is None or len(sc) == 0:  # last-resort fallback: std over the evaluation predictions
        sc = (predictions_df.dropna(subset=[TRUE_COL, TICKER_COL])
              .groupby(TICKER_COL)[TRUE_COL].std().rename('pnl_std').reset_index()
              .rename(columns={TICKER_COL: 'ticker'}))
    sc['log_pnl_std'] = np.log(sc['pnl_std'].replace(0, np.nan))
    d = detail.merge(sc[['ticker', 'log_pnl_std']], on='ticker', how='left')
    rows = []
    for gv, g in d.groupby(keys_noyear, dropna=False):
        base = dict(zip(keys_noyear, gv if isinstance(gv, tuple) else (gv,)))
        rows.append({**base,
                     'pearson_r': safe_pearson(g['within_ticker_spearman'], g['log_pnl_std']),
                     'n_tickers': int((g['within_ticker_spearman'].notna() & g['log_pnl_std'].notna()).sum())})
    t = pd.DataFrame(rows)
    register_table('within_ranking_vs_pnl_scale', t)
    return t




## 14. One-click execution

Selects the four targets, the single `final_feature_b` set, and the LightGBM + baselines model list, then runs the walk-forward experiments.

In [ ]:
# Binary targets for within-ticker ROC-AUC 
BINARY_TARGETS = ['rl_long_action_binary', 'pi_hindsight_entry_positive']
# Recommended walk-forward run:
EXPERIMENT_CONFIG['target_names'] = ['rl_long_current_pnl', 'rl_long_action_binary', 'rl_long_action_quality', 'pi_hindsight_entry_positive']  # hindsight: positive/zero only (dropped >=0.4 and continuous)
EXPERIMENT_CONFIG['feature_set_names'] = ['final_feature_b']
EXPERIMENT_CONFIG['model_names'] = ['DummyMean', 'DummyMostFrequent', 'ElasticNet', 'Logistic', 'RandomForest', 'LightGBM']

# Yearly retraining designs requested for the final experiment.
WALK_FORWARD_CONFIG['enabled'] = True
WALK_FORWARD_CONFIG['schemes'] = ['expanding', 'rolling_2y']
WALK_FORWARD_CONFIG['rolling_window_years'] = 2
WALK_FORWARD_CONFIG['min_train_years'] = 2
WALK_FORWARD_CONFIG['retrain_frequency'] = 'yearly'




# Main requested experiment: expanding yearly + rolling 2-year yearly walk-forward.
metrics_df, predictions_df, skipped_df = run_walk_forward_experiments(
    frames,
    FEATURE_SET_CONFIGS,
    target_configs=TARGET_CONFIGS,
    config=EXPERIMENT_CONFIG,
    walk_config=WALK_FORWARD_CONFIG,
)

# Standard diagnostics: overall metrics, raw daily cross-sectional Spearman, Top-K,
# within-ticker ranking, subgroup diagnostics, and calibration.
diagnostic_tables = run_prediction_diagnostics(predictions_df)

# Additional requested diagnostic:
# convert each prediction score to a historical within-ticker percentile and then
# evaluate daily cross-sectional Spearman using that percentile score.
percentile_diagnostic_tables = run_within_ticker_percentile_diagnostics(predictions_df)
scale_neutral_cross_sectional_diagnostics(predictions_df)
descaled_topk_diagnostics(predictions_df)
scale_neutral_decile_diagnostics(predictions_df)
within_ticker_spread_by_fold(predictions_df)
within_ticker_decile_diagnostics(predictions_df)
ticker_heterogeneity_diagnostics(predictions_df)
within_ticker_rocauc(predictions_df)
within_ranking_vs_pnl_scale(predictions_df)
ranking_confidence_intervals(predictions_df)
signedlog_error_metrics(predictions_df)
plot_within_ticker_decile(predictions_df)
diagnostic_tables.update(percentile_diagnostic_tables)

best_model_tables = build_best_model_tables(metrics_df)
best_ranking_tables = build_best_ranking_tables()

# Current-PnL deciles are only meaningful if rl_long_current_pnl is included in EXPERIMENT_CONFIG['target_names'].
current_pnl_deciles, current_pnl_deciles_by_year = temporal_decile_analysis(predictions_df)

#grouped_metrics, grouped_predictions = run_grouped_model_experiments(frames, FEATURE_SET_CONFIGS)

# Simulator-compatible export is designed around named static splits such as split='test'.
# Walk-forward row-level predictions are still exported as parquet by run_walk_forward_experiments.
prediction_submission_paths = []

Walk-forward folds: 10


,walkforward_scheme,fold_id,train_start_year,train_end_year,n_train_years,test_year
0,expanding,expanding_test_2022,2020,2021,2,2022
1,expanding,expanding_test_2023,2020,2022,3,2023
2,expanding,expanding_test_2024,2020,2023,4,2024
3,expanding,expanding_test_2025,2020,2024,5,2025
4,expanding,expanding_test_2026,2020,2025,6,2026
5,rolling_2y,rolling_2y_test_2022,2020,2021,2,2022
6,rolling_2y,rolling_2y_test_2023,2021,2022,2,2023
7,rolling_2y,rolling_2y_test_2024,2022,2023,2,2024
8,rolling_2y,rolling_2y_test_2025,2023,2024,2,2025
9,rolling_2y,rolling_2y_test_2026,2024,2025,2,2026


WF: scheme=expanding | test_year=2022 | target=pi_hindsight_entry_positive | features=final_feature_b | model=DummyMostFrequent
WF: scheme=expanding | test_year=2023 | target=pi_hindsight_entry_positive | features=final_feature_b | model=DummyMostFrequent
WF: scheme=expanding | test_year=2024 | target=pi_hindsight_entry_positive | features=final_feature_b | model=DummyMostFrequent
WF: scheme=expanding | test_year=2025 | target=pi_hindsight_entry_positive | features=final_feature_b | model=DummyMostFrequent
WF: scheme=expanding | test_year=2026 | target=pi_hindsight_entry_positive | features=final_feature_b | model=DummyMostFrequent
WF: scheme=rolling_2y | test_year=2022 | target=pi_hindsight_entry_positive | features=final_feature_b | model=DummyMostFrequent
WF: scheme=rolling_2y | test_year=2023 | target=pi_hindsight_entry_positive | features=final_feature_b | model=DummyMostFrequent
WF: scheme=rolling_2y | test_year=2024 | target=pi_hindsight_entry_positive | features=final_feature_b

## 12.3  Within-ticker Spearman with dependence-robust CIs (block bootstrap)

Point estimate follows Fama–MacBeth: within each fold, the mean over tickers of the
per-ticker Spearman(score, true); averaged over the full-year folds (partial years excluded).
Confidence intervals use a **fold-stratified, date-clustered moving-block bootstrap** — within
each fold, contiguous 21-trading-day blocks of dates are resampled with replacement and each
sampled day's full cross-section is kept together; blocks do not cross retrain boundaries.
`B = 1000`. Registers `within_ticker_spearman_blockboot_ci` (fold-average row + per-fold rows).


In [27]:
# ============================================================
# 12.3  Per-fold within-ticker Spearman + fold-stratified, date-clustered
#       moving-block bootstrap CIs.  Registers 'within_ticker_spearman_blockboot_ci'.
# ============================================================
import numpy as np

WT_CI_BLOCK, WT_CI_B, WT_CI_MIN_ROWS = 21, 1000, 20

def _wtci_avg_rank(a):
    n = a.size
    order = np.argsort(a, kind='mergesort'); sa = a[order]
    grp = np.cumsum(np.r_[True, sa[1:] != sa[:-1]]) - 1
    avg = np.bincount(grp, weights=np.arange(1, n + 1, dtype=float)) / np.bincount(grp)
    r = np.empty(n); r[order] = avg[grp]
    return r

def _wtci_rho(t, s, min_valid=3):
    m = np.isfinite(t) & np.isfinite(s)
    if m.sum() < min_valid: return np.nan
    t, s = t[m], s[m]
    if np.unique(t).size < 2 or np.unique(s).size < 2: return np.nan
    rt, rs = _wtci_avg_rank(t), _wtci_avg_rank(s)
    rt -= rt.mean(); rs -= rs.mean()
    d = np.sqrt((rt @ rt) * (rs @ rs))
    return float((rt @ rs) / d) if d > 0 else np.nan

def _wtci_mean_over_tickers(t, s, codes, min_rows):
    order = np.argsort(codes, kind='mergesort')
    c, ts, ss = codes[order], t[order], s[order]
    bnd = np.flatnonzero(np.r_[True, c[1:] != c[:-1], True]); out = []
    for i in range(bnd.size - 1):
        a, b = bnd[i], bnd[i + 1]
        if b - a >= min_rows:
            r = _wtci_rho(ts[a:b], ss[a:b])
            if np.isfinite(r): out.append(r)
    return np.mean(out) if out else np.nan

def _wtci_partial_years(df, year_key, ratio=0.75):
    if year_key is None or DATE_COL not in df.columns: return set()
    nd = df.dropna(subset=[year_key]).groupby(year_key)[DATE_COL].nunique()
    if len(nd) == 0: return set()
    return set(nd[nd < ratio * nd.median()].index)

def within_ticker_spearman_blockboot(predictions_df, B=WT_CI_B, block=WT_CI_BLOCK,
                                     min_ticker_rows=WT_CI_MIN_ROWS, seed=0, targets=None):
    """Per-fold within-ticker Spearman (fold-averaged) with fold-stratified,
    date-clustered moving-block bootstrap 95% CIs. Rank-based; suitable for
    continuous or binary targets (for binary the within-ticker Spearman is
    the within-ticker AUC-equivalent)."""
    if predictions_df is None or len(predictions_df) == 0: return {}
    df = predictions_df.dropna(subset=[SCORE_COL, TRUE_COL, TICKER_COL, DATE_COL]).copy()
    year_key = 'test_year' if 'test_year' in df.columns else (YEAR_COL if YEAR_COL in df.columns else None)
    gcols = [c for c in ['target_name', 'feature_set', 'model', 'walkforward_scheme'] if c in df.columns]
    partial = _wtci_partial_years(predictions_df, year_key)
    rows = []
    for keys, g in df.groupby(gcols, dropna=False):
        base = dict(zip(gcols, keys if isinstance(keys, tuple) else (keys,)))
        if targets is not None and base.get('target_name') not in targets: continue
        gg = g[~g[year_key].isin(partial)] if year_key else g
        folds = sorted(gg[year_key].dropna().unique().tolist()) if year_key else [None]
        fdata = []
        for fy in folds:
            gf = gg[gg[year_key] == fy] if year_key else gg
            t = pd.to_numeric(gf[TRUE_COL], errors='coerce').to_numpy(float)
            s = pd.to_numeric(gf[SCORE_COL], errors='coerce').to_numpy(float)
            tk = pd.factorize(gf[TICKER_COL].to_numpy())[0]
            dc, du = pd.factorize(gf[DATE_COL].to_numpy(), sort=True)
            order = np.argsort(dc, kind='mergesort'); cc = dc[order]
            bnd = np.flatnonzero(np.r_[True, cc[1:] != cc[:-1], True])
            pos = [order[bnd[i]:bnd[i+1]] for i in range(bnd.size - 1)]
            fdata.append(dict(y=fy, t=t, s=s, tk=tk, pos=pos, nd=len(du)))
        per_fold_pt = [_wtci_mean_over_tickers(fd['t'], fd['s'], fd['tk'], min_ticker_rows) for fd in fdata]
        theta = float(np.nanmean(per_fold_pt)) if per_fold_pt else np.nan
        rng = np.random.default_rng(seed)
        dfold = [[] for _ in fdata]; dth = []
        for _ in range(B):
            ms = []
            for j, fd in enumerate(fdata):
                nd = fd['nd']
                if nd == 0: dfold[j].append(np.nan); ms.append(np.nan); continue
                starts = np.arange(0, max(1, nd - block + 1))
                seq = np.concatenate([np.arange(c, min(c + block, nd))
                                      for c in rng.choice(starts, size=int(np.ceil(nd / block)), replace=True)])
                idx = np.concatenate([fd['pos'][c] for c in seq])
                v = _wtci_mean_over_tickers(fd['t'][idx], fd['s'][idx], fd['tk'][idx], min_ticker_rows)
                dfold[j].append(v); ms.append(v)
            dth.append(np.nanmean(ms))
        def _ci(a):
            a = np.array(a, float); a = a[np.isfinite(a)]
            return (np.nan, np.nan) if a.size == 0 else (float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5)))
        lo, hi = _ci(dth)
        rows.append({**base, 'test_year': 'fold_average', 'n_folds': len(fdata),
                     'within_ticker_spearman': round(theta, 4), 'ci_low': round(lo, 4), 'ci_high': round(hi, 4),
                     'B': B, 'block': block})
        for j, fd in enumerate(fdata):
            flo, fhi = _ci(dfold[j])
            rows.append({**base, 'test_year': fd['y'], 'n_folds': 1,
                         'within_ticker_spearman': (round(per_fold_pt[j], 4) if per_fold_pt[j] == per_fold_pt[j] else np.nan),
                         'ci_low': (round(flo, 4) if flo == flo else np.nan),
                         'ci_high': (round(fhi, 4) if fhi == fhi else np.nan), 'B': B, 'block': block})
    out = pd.DataFrame(rows)
    register_table('within_ticker_spearman_blockboot_ci', out)
    return out

# Run it (B=1000). Dummy baselines yield NaN (constant prediction) and are harmless.
within_ticker_ci = within_ticker_spearman_blockboot(predictions_df, B=1000)
display(within_ticker_ci[within_ticker_ci['test_year'] == 'fold_average'])


,target_name,feature_set,model,walkforward_scheme,test_year,n_folds,within_ticker_spearman,ci_low,ci_high,B,block
0,pi_hindsight_entry_positive,final_feature_b,DummyMostFrequent,expanding,fold_average,4,NaN,NaN,NaN,1000,21
5,pi_hindsight_entry_positive,final_feature_b,DummyMostFrequent,rolling_2y,fold_average,4,NaN,NaN,NaN,1000,21
10,pi_hindsight_entry_positive,final_feature_b,LightGBM,expanding,fold_average,4,0.1422,0.0863,0.1814,1000,21
15,pi_hindsight_entry_positive,final_feature_b,LightGBM,rolling_2y,fold_average,4,0.1324,0.0762,0.1713,1000,21
20,pi_hindsight_entry_positive,final_feature_b,Logistic,expanding,fold_average,4,0.1377,0.0865,0.1889,1000,21
25,pi_hindsight_entry_positive,final_feature_b,Logistic,rolling_2y,fold_average,4,0.1136,0.0706,0.1741,1000,21
30,pi_hindsight_entry_positive,final_feature_b,RandomForest,expanding,fold_average,4,0.1670,0.1173,0.2059,1000,21
35,pi_hindsight_entry_positive,final_feature_b,RandomForest,rolling_2y,fold_average,4,0.1507,0.1015,0.1873,1000,21
40,rl_long_action_binary,final_feature_b,DummyMostFrequent,expanding,fold_average,4,NaN,NaN,NaN,1000,21
45,rl_long_action_binary,final_feature_b,DummyMostFrequent,rolling_2y,fold_average,4,NaN,NaN,NaN,1000,21


## 13.2 Current-PnL robustness checks (heavy tail)

Because `rl_long_current_pnl` is the heavy-tailed primary target, we can test whether the ranking result depends on the tail. Two checks:

1. **Signed-log target** `y* = sign(y)·log(1+|y|)`: retrain on the tamed target and compare Spearman with the raw target. **Top-K is still scored on RAW PnL** (joined back via `IndexReference`), so we still see whether the model captures the big winners. Signed-log retraining is implemented in the dedicated signed-log notebook; only extreme-value sensitivity is executed here
2. **Extreme-value sensitivity**: recompute pooled Spearman on the full sample vs after **excluding the top/bottom 0.1%** vs **winsorising at 1/99%** — does the ranking value collapse without a few extreme observations?

*Applicable label:* `rl_long_current_pnl` only.

In [28]:
# ---- Robustness 1: signed-log current PnL (ranking comparison + Top-K on RAW PnL) ----
def _rank_summary(pred, target):
    d = pred[pred['target_name'] == target]
    rows = []
    for m, g in d.groupby('model'):
        wt = within_ticker_ranking(g)
        rows.append({'target': target, 'model': m,
                     'pooled_spearman': safe_spearman(g[TRUE_COL], g[SCORE_COL]),
                     'within_ticker_median_spearman':
                         wt['within_ticker_spearman'].median() if len(wt) else np.nan})
    return pd.DataFrame(rows)

RUN_SIGNEDLOG_ROBUSTNESS = False   # set True to also train a signed-log model (transform-invariance check)
try:
    assert RUN_SIGNEDLOG_ROBUSTNESS, 'RUN_SIGNEDLOG_ROBUSTNESS is False; skipping the signed-log-trained robustness check'
    _protect = ['walk_forward_model_metric_summary', 'walk_forward_skipped_or_failed_experiments', 'walk_forward_fold_definition']
    _backup = {_k: EXPORTED_TABLES.get(_k) for _k in _protect}
    _cfg = dict(EXPERIMENT_CONFIG); _cfg['target_names'] = ['rl_long_current_pnl_signedlog']
    sl_metrics, sl_pred, _ = run_walk_forward_experiments(
        frames, FEATURE_SET_CONFIGS, target_configs=TARGET_CONFIGS,
        config=_cfg, walk_config=WALK_FORWARD_CONFIG)
    register_table('walk_forward_model_metric_summary_signedlog', sl_metrics)
    for _k, _v in _backup.items():
        if _v is not None:
            register_table(_k, _v)
    signedlog_ranking = pd.concat([
        _rank_summary(predictions_df, 'rl_long_current_pnl'),
        _rank_summary(sl_pred, 'rl_long_current_pnl_signedlog')], ignore_index=True)
    register_table('robustness_signedlog_ranking', signedlog_ranking); display(signedlog_ranking)

    # Top-K scored on RAW PnL: rank by the signed-log prediction, but evaluate actual raw PnL.
    raw_pnl = all_data[[INDEX_COL, 'target__rl_long_current_pnl']].rename(
        columns={'target__rl_long_current_pnl': '_raw_pnl'})
    sl_join = sl_pred.merge(raw_pnl, on=INDEX_COL, how='left').copy()
    sl_join[TRUE_COL] = sl_join['_raw_pnl']
    topk_rows = []
    for m in sl_join['model'].dropna().unique():
        for p in EXPERIMENT_CONFIG['topk_percentages']:
            t = topk_diagnostics(sl_join[sl_join['model'] == m], top_pct=p)
            if len(t):
                t['model'] = m; topk_rows.append(t)
    if topk_rows:
        register_table('robustness_signedlog_topk_on_raw', pd.concat(topk_rows, ignore_index=True))
    print('Signed-log robustness done (main tables preserved; signed-log metrics under *_signedlog).')
except Exception as e:
    print('Signed-log robustness skipped:', e)


# ---- Robustness 2: extreme-value sensitivity on raw current PnL ----
def extreme_value_sensitivity(pred, target='rl_long_current_pnl'):
    d = pred[pred['target_name'] == target].dropna(subset=[TRUE_COL, SCORE_COL])
    rows = []
    for m, g in d.groupby('model'):
        y = pd.to_numeric(g[TRUE_COL], errors='coerce'); s = pd.to_numeric(g[SCORE_COL], errors='coerce')
        lo, hi = y.quantile(0.001), y.quantile(0.999)          # exclude most extreme 0.1% tails
        keep = (y >= lo) & (y <= hi)
        w = y.clip(y.quantile(0.01), y.quantile(0.99))          # winsorise at 1/99%
        rows.append({'target': target, 'model': m, 'n': int(len(g)),
                     'spearman_full': safe_spearman(y, s),
                     'spearman_excl_0.1pct_tails': safe_spearman(y[keep], s[keep]),
                     'spearman_winsor_1_99': safe_spearman(w, s)})
    return pd.DataFrame(rows)

try:
    ev = extreme_value_sensitivity(predictions_df)
    register_table('robustness_extreme_value', ev); display(ev)
except Exception as e:
    print('Extreme-value sensitivity skipped:', e)

Signed-log robustness skipped: RUN_SIGNEDLOG_ROBUSTNESS is False; skipping the signed-log-trained robustness check


,target,model,n,spearman_full,spearman_excl_0.1pct_tails,spearman_winsor_1_99
0,rl_long_current_pnl,DummyMean,210388,-0.059069,-0.056205,-0.059049
1,rl_long_current_pnl,ElasticNet,210388,0.056919,0.058146,0.056897
2,rl_long_current_pnl,LightGBM,210388,0.146089,0.148106,0.146051
3,rl_long_current_pnl,RandomForest,210388,0.161144,0.160571,0.161082


In [29]:
# ============================================================
# Final Figure Export for the dissertation -> saved to FIGURE_DIR
# Reads the registered diagnostic tables (EXPORTED_TABLES); run after the
# diagnostics block. No captions are drawn on the figures.
# ============================================================
def make_maintext_figures(target='rl_long_current_pnl', scheme='rolling_2y'):
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    from matplotlib.ticker import PercentFormatter
    import numpy as np, pandas as pd
    T = EXPORTED_TABLES
    def get(n):
        t = T.get(n); return t.copy() if t is not None else None
    cmap = {'ElasticNet': '#3b6fb0', 'LightGBM': '#f0a500', 'RandomForest': '#2e7d32'}
    SCATTER_BLUE = '#3b6fb0'
    order = ['ElasticNet', 'LightGBM', 'RandomForest']
    yrs = [2022, 2023, 2024, 2025]

    # ---- Figure 5.1: within-ticker Spearman by fold ----
    wt = get('within_ticker_summary')
    if wt is not None:
        d = wt[(wt['target_name'] == target) & (wt['walkforward_scheme'] == scheme) & (wt['test_year'].isin(yrs))]
        plt.figure(figsize=(6, 4))
        for mo in order:
            g = d[d['model'] == mo].sort_values('test_year')
            if len(g):
                plt.plot(g['test_year'], g['mean_within_ticker_spearman'], marker='o', ms=4, color=cmap[mo], label=mo)
        plt.axhline(0, c='0.8', lw=.8); plt.xticks(yrs)
        plt.xlabel('Test year'); plt.ylabel('Within-ticker Spearman')
        plt.title('Within-ticker Spearman by fold'); plt.legend(frameon=False, fontsize=8.5); plt.grid(alpha=.2)
        plt.tight_layout(); plt.savefig(FIGURE_DIR / 'fig_within_spearman_by_fold.png', dpi=300, bbox_inches='tight'); plt.close()

    # ---- shared 2-panel decile figure ----
    def decile_fig(df, xlabel, pctlabel, out):
        base_rate = float(np.average(df['positive_rate'], weights=df['_w']))
        fig, ax = plt.subplots(1, 2, figsize=(10.5, 4.3))
        handles = []
        for mo in order:
            g = df[df['model'] == mo].sort_values('decile')
            if not len(g): continue
            l, = ax[0].plot(g['decile'], g['mean_realised_pct'], marker='o', ms=4, color=cmap[mo], label=mo)
            ax[1].plot(g['decile'], g['positive_rate'], marker='o', ms=4, color=cmap[mo])
            handles.append(l)
        ax[0].axhline(0.5, ls='--', c='0.6', lw=1)
        ax[1].axhline(base_rate, ls='--', c='0.6', lw=1)
        ax[1].annotate('base rate %.0f%%' % (base_rate * 100), (1, base_rate),
                       xytext=(1.05, base_rate + 0.012), fontsize=8, color='0.4')
        ax[0].set_title('(a) Mean realised percentile', fontsize=11); ax[0].set_ylabel(pctlabel)
        ax[1].set_title('(b) Positive-PnL rate', fontsize=11); ax[1].set_ylabel('Positive-PnL rate')
        ax[0].set_ylim(0.35, 0.65); ax[1].set_ylim(0.25, 0.70)
        ax[1].yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
        for a in ax:
            a.set_xlabel(xlabel); a.set_xticks(range(1, 11)); a.grid(alpha=.2)
        ax[0].legend(handles=handles, loc='upper left', frameon=False, fontsize=8.5)
        plt.tight_layout(); plt.savefig(FIGURE_DIR / out, dpi=300, bbox_inches='tight'); plt.close()

    # within-ticker decile (already pooled over folds)
    wd = get('within_ticker_decile_pooled')
    if wd is not None:
        d = wd[(wd['target_name'] == target) & (wd['walkforward_scheme'] == scheme)].copy()
        d['_w'] = d['n']
        decile_fig(d[['model', 'decile', 'mean_realised_pct', 'positive_rate', '_w']],
                   'Within-ticker prediction decile', 'Mean realised within-ticker percentile',
                   'fig_within_decile.png')

    # cross-sectional decile (average over folds 2022-2025)
    cd = get('scaleneutral_cross_sectional_decile')
    if cd is not None:
        d = cd[(cd['target_name'] == target) & (cd['walkforward_scheme'] == scheme) & (cd['test_year'].isin(yrs))].copy()
        w = 'n_days' if 'n_days' in d.columns else None
        if w is None:
            d['n_days'] = 1.0
        d['_wp'] = d['mean_realised_pct'] * d['n_days']; d['_wr'] = d['positive_rate'] * d['n_days']
        agg = d.groupby(['model', 'decile']).agg(_wp=('_wp', 'sum'), _wr=('_wr', 'sum'), _w=('n_days', 'sum')).reset_index()
        agg['mean_realised_pct'] = agg['_wp'] / agg['_w']; agg['positive_rate'] = agg['_wr'] / agg['_w']
        decile_fig(agg[['model', 'decile', 'mean_realised_pct', 'positive_rate', '_w']],
                   'Cross-sectional prediction decile', 'Mean realised ticker-normalised percentile',
                   'fig_cross_decile.png')

    # ---- Section 4.3 heterogeneity ----
    det = get('ticker_heterogeneity_detail'); tsb = get('target_summary_by_ticker'); cmt = get('cross_model_within_ticker_consistency')
    if det is not None:
        d = det[(det['target_name'] == target) & (det['walkforward_scheme'] == scheme)]
        rf = d[d['model'] == 'RandomForest'].set_index('ticker')['within_ticker_spearman']
        lg = d[d['model'] == 'LightGBM'].set_index('ticker')['within_ticker_spearman']
        common = rf.index.intersection(lg.index)
        rho = np.nan
        if cmt is not None:
            rr = cmt[(cmt['target_name'] == target) & (cmt['walkforward_scheme'] == scheme) & (cmt['model_pair'] == 'RandomForest vs LightGBM')]
            if len(rr): rho = float(rr['spearman_rho'].iloc[0])
        # Figure 4.3(a): cross-model consistency scatter (RF vs LGBM), key tickers labelled
        dev = [tk for tk in ['UNH', 'ORCL', 'SLB', 'BRK.B'] if tk in rf.index and tk in lg.index]
        plt.figure(figsize=(5.6, 5.2))
        plt.scatter(rf[common], lg[common], s=20, alpha=.75, color=SCATTER_BLUE)
        lim = [-0.12, 0.6]; plt.plot(lim, lim, '--', c='0.5', lw=1, label='y = x')
        plt.axhline(0, c='0.85', lw=.8); plt.axvline(0, c='0.85', lw=.8)
        for t in dev:
            plt.annotate(str(t), (rf[t], lg[t]), xytext=(4, 3), textcoords='offset points', fontsize=8, color='#c0392b')
        plt.xlim(lim); plt.ylim(-0.12, 0.58)
        plt.xlabel('RandomForest per-ticker Spearman'); plt.ylabel('LightGBM per-ticker Spearman')
        plt.title(u'(a) Cross-model consistency in per-ticker performance (\u03c1 = %.2f)' % rho, fontsize=10.5)
        plt.legend(loc='upper left', frameon=False, fontsize=8.5)
        plt.tight_layout(); plt.savefig(FIGURE_DIR / 'fig_4_3a_rf_vs_lgbm.png', dpi=300, bbox_inches='tight'); plt.close()
        # Figure 4.3(b): top & bottom 10 tickers by RF (green/red + legend)
        from matplotlib.patches import Patch
        o = rf.sort_values(); sel = pd.concat([o.tail(10).iloc[::-1], o.head(10).iloc[::-1]])
        plt.figure(figsize=(5.6, 6))
        plt.barh(range(len(sel)), sel.values, color=['#2e7d32'] * 10 + ['#c0392b'] * 10)
        plt.gca().invert_yaxis()
        plt.yticks(range(len(sel)), sel.index, fontsize=8); plt.axvline(0, c='0.4', lw=.8)
        plt.xlabel('Within-ticker Spearman (RandomForest)'); plt.title('Top and bottom 10 tickers by RandomForest', fontsize=10.5)
        plt.legend(handles=[Patch(color='#2e7d32', label='Top 10'), Patch(color='#c0392b', label='Bottom 10')], loc='lower right', frameon=True, fontsize=8.5)
        plt.tight_layout(); plt.savefig(FIGURE_DIR / 'fig_4_3b_top_bottom.png', dpi=300, bbox_inches='tight'); plt.close()
        # Figure C: sector heterogeneity, one figure per model (main text: RF; appendix: LGBM)
        if 'gics_industry' in d.columns:
            dot_col = {'RandomForest': cmap['RandomForest'], 'LightGBM': cmap['LightGBM']}
            dia_col = {'RandomForest': '#14532d', 'LightGBM': '#9a6b00'}  # darker same-hue for the sector mean
            title_nm = {'RandomForest': 'Random Forest', 'LightGBM': 'LightGBM'}
            for mo, ser in [('RandomForest', rf), ('LightGBM', lg)]:
                dd = d[d['model'] == mo][['ticker', 'within_ticker_spearman', 'gics_industry']].dropna(subset=['gics_industry'])
                grp = dd.groupby('gics_industry')['within_ticker_spearman']
                secs = [(s, g.mean(), g.values) for s, g in grp if g.count() >= 5]; secs.sort(key=lambda z: z[1])
                overall = float(ser.mean())
                fig, axc = plt.subplots(figsize=(6.8, 5))
                for i, (s, mn, v) in enumerate(secs):
                    axc.scatter(v, [i] * len(v), s=20, alpha=.6, color=dot_col[mo])
                    axc.scatter([mn], [i], marker='D', s=60, color=dia_col[mo], zorder=3)
                axc.axvline(0.0, ls='--', c='0.5', lw=.9)
                axc.axvline(overall, ls=':', c='0.45', lw=1.1)
                axc.annotate('Overall mean', xy=(overall, -0.5), xytext=(4, 0), textcoords='offset points',
                             va='center', ha='left', fontsize=7.5, color='0.35')
                axc.set_yticks(range(len(secs))); axc.set_yticklabels([s for s, mn, v in secs], fontsize=9)
                axc.set_xlim(-0.2, 0.72); axc.set_ylim(-0.6, len(secs) - 0.4)
                for i, (s, mn, v) in enumerate(secs):
                    axc.annotate('n=%d' % len(v), xy=(0.985, i), xycoords=('axes fraction', 'data'),
                                 ha='right', va='center', fontsize=8, color='0.45')
                axc.set_xlabel('Within-ticker Spearman correlation'); axc.set_title('Ticker heterogeneity by sector (%s)' % title_nm[mo], fontsize=11)
                plt.tight_layout(); plt.savefig(FIGURE_DIR / ('fig_C_sector_%s.png' % mo), dpi=300, bbox_inches='tight'); plt.close()
        # Figure 4.X: PnL scale, two panels (LightGBM, RandomForest), initial-training-window std
        std = None
        _sc = _pnl_scale_initial_window('target__rl_long_current_pnl')
        if _sc is not None and len(_sc):
            std = _sc.set_index('ticker')['pnl_std'].astype(float)
        if std is not None:
            fig, ax = plt.subplots(1, 2, figsize=(10.5, 4.3), sharex=True, sharey=True)
            for a, (ttl, ser) in zip(ax, [('(a) LightGBM', lg), ('(b) Random Forest', rf)]):
                idx = ser.index.intersection(std.index)
                x = np.log10(std[idx]); y = ser[idx].astype(float); m = np.isfinite(x) & np.isfinite(y)
                x = np.asarray(x[m], float); y = np.asarray(y[m], float)
                r = np.corrcoef(x, y)[0, 1]
                a.scatter(x, y, s=18, alpha=.7, color=SCATTER_BLUE)
                b1, b0 = np.polyfit(x, y, 1); xs = np.array([x.min(), x.max()])
                a.plot(xs, b0 + b1 * xs, color='0.35', lw=1.5)
                a.axhline(0, ls=':', c='0.7', lw=.8)
                a.set_title(ttl, fontsize=11)
                a.annotate('Pearson r = %.2f' % r, xy=(0.03, 0.96),
                           xycoords='axes fraction', va='top', fontsize=8.5, color='0.3')
                a.set_xlabel('Log10 ticker-level current-PnL standard deviation')
            ax[0].set_ylabel('Fold-averaged within-ticker Spearman')
            ax[0].set_xlim(2.4, 4.35); ax[0].set_ylim(-0.15, 0.58)
            plt.tight_layout(); plt.savefig(FIGURE_DIR / 'fig_4_X_pnl_scale.png', dpi=300, bbox_inches='tight'); plt.close()
    print('Main-text figures saved to:', FIGURE_DIR)


make_maintext_figures()


PnL scale: std over initial training window [2020, 2021] (98 tickers, point-in-time, no look-ahead).
Main-text figures saved to: c:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment_0806\integrated_experiment_outputs_recent_final_feature_b\figures


In [30]:
final_workbook_path = export_registered_tables()
final_workbook_path

Integrated workbook exported to: C:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment_0806\integrated_experiment_outputs_recent_final_feature_b\tables\integrated_experiment_tables_20260808_042747.xlsx
Number of registered tables: 45
Number of non-empty tables written: 43


,table_name,reason
0,feature_leakage_check,empty_dataframe
1,walk_forward_skipped_or_failed_experiments,empty_dataframe


WindowsPath('c:/Users/user/Downloads/universe_100_recent_post_normalisation_experiment_0806/integrated_experiment_outputs_recent_final_feature_b/tables/integrated_experiment_tables_20260808_042747.xlsx')

## 15. Checklist

In [31]:
def build_notebook_checklist():
    rows = [
        ('0. Environment setup', 'Single install + single import/config cell: imports, random seed, canonical column names (IndexReference, attr__ticker, attr__timestamp, year, split), output directories, and signal-interface columns.', 'Keeps variables consistent across the notebook and preserves the downstream simulator join key.'),
        ('1. Dataset configuration and loading', 'DATASET_CONFIG drives Universe/time-horizon/normalisation paths; JSONL/parquet/csv/zip loaders flatten records to attr__/feature__/label__ and standardise train/valid/test frames (year derived from timestamp). Includes register_table / export manager.', 'Allows dataset swaps without rewriting modelling code and exports every table reliably.'),
        ('2. Label construction and target registry', 'Builds target__ columns (pi_hindsight entry long/positive/6bins, RL long action/quality/binary, current PnL) and the TARGET_CONFIGS task-type registry.', 'Makes label definitions explicit and reusable across models.'),
        ('3. Feature inventory for modelling', 'Inventories numeric modelling-feature candidates; full feature EDA now lives in the separate EDA notebook.', 'Keeps this notebook focused on modelling and out of the EDA/look-ahead surface.'),
        ('4. Final feature set (FINAL_FEATURE_B) + leakage check', 'Fixes the 268-feature final_feature_b set and hard-fails on any label__/target__ leakage, flagging keyword matches (reward/pnl/future/...) for manual review.', 'Guarantees modelling features contain no outcome or forward-looking information.'),
        ('5. Shared modelling utilities and models', 'Preprocessing (median impute; scaling fit inside each fold for linear models only, none for trees), model zoo (Dummy, Ridge/Lasso/ElasticNet, Logistic, RandomForest, HistGB, LightGBM), and the prediction-frame builder.', 'Reuses one consistent pipeline and prepares simulator-ready signal outputs.'),
        ('6. Metric functions', 'Regression, binary, multiclass, calibration, daily cross-sectional ranking, Top-K, within-ticker, and subgroup metrics, with positional y/pred pairing to avoid index misalignment.', 'Provides the full metric set the diagnostics and comparison tables depend on.'),
        ('7. Dependence-robust inference layer', 'Spearman with Newey-West and moving-block-bootstrap-over-dates CIs, classification metrics with block-bootstrap CIs, and paired model comparison with two-sided bootstrap p-values.', 'Corrects for cross-sectional and serial dependence so confidence intervals are not overstated.'),
        ('8. Main experiment runner (fixed train/valid/test)', 'Loops targets x feature sets x models on the static split with IndexReference-preserving predictions and exports row-level prediction parquet.', 'Runs the static-split baseline from one auditable framework.'),
        ('8.1 Simulator-compatible prediction export', 'Structured JSON/JSONL export with trade.long / trade.short / trade.no_trade and signal.score heads.', 'Produces downstream trade-simulator-compatible prediction files.'),
        ('9. Post-model diagnostics', 'Daily Spearman, Top-K, within-ticker ranking, scale-neutral cross-sectional & decile, by-year/by-ticker/by-SIC2 subgroups, and calibration.', 'Characterises ranking quality beyond a single headline number.'),
        ('10. Best-model comparison tables', 'Best-model, feature-family, best-ranking, and best-Top-K summary tables.', 'Supports concise dissertation interpretation.'),
        ('11. Current-PnL temporal decile analysis', 'Prediction-decile realised-outcome tables overall and by year for rl_long_current_pnl.', 'Connects model scores to realised trading-outcome quality over time.'),
        ('12. Yearly walk-forward (expanding & rolling 2-year)', 'Yearly retraining folds: expanding = all prior years, rolling_2y = last 2 years before each test year; scores mapped to percentiles using the training-window distribution only.', 'Out-of-sample, point-in-time evaluation matching a realistic yearly retraining cadence.'),
        ('12.3 Within-ticker heterogeneity, sectors & PnL scale', 'Fold-averaged per-ticker within-ticker Spearman, SIC2/GICS sector aggregation, within-ticker ROC-AUC, dependence-robust CIs, and the ranking-vs-PnL-scale relationship (scale measured over the initial 2-year training window, no look-ahead).', 'Shows where ranking skill concentrates across tickers/sectors and whether it depends on PnL magnitude.'),
        ('13.2 Current-PnL robustness checks (heavy tail)', 'Signed-log target ranking comparison (Top-K still scored on RAW PnL) and extreme-value sensitivity: full vs excluding the 0.1% tails vs 1/99% winsorised.', 'Confirms the ranking result does not hinge on a few heavy-tail observations.'),
        ('14. One-click execution + main-text figures', 'Full execution block plus saved Section 4.2/4.3 figures: within-ticker Spearman by fold, decile panels, cross-model consistency, sector heterogeneity, and the PnL-scale scatter.', 'Makes the notebook rerunnable/auditable and regenerates every main-text figure.'),
        ('15. Checklist', 'This table: what each section does and why it matters.', 'Gives reviewers a one-glance map of the pipeline.'),
    ]
    checklist = pd.DataFrame(rows, columns=['section', 'task_completed', 'why_it_matters'])
    register_table('notebook_task_checklist', checklist)
    return checklist

notebook_task_checklist = build_notebook_checklist()
notebook_task_checklist

,section,task_completed,why_it_matters
0,0. Environment setup,Single install + single import/config cell: im...,Keeps variables consistent across the notebook...
1,1. Dataset configuration and loading,DATASET_CONFIG drives Universe/time-horizon/no...,Allows dataset swaps without rewriting modelli...
2,2. Label construction and target registry,Builds target__ columns (pi_hindsight entry lo...,Makes label definitions explicit and reusable ...
3,3. Feature inventory for modelling,Inventories numeric modelling-feature candidat...,Keeps this notebook focused on modelling and o...
4,4. Final feature set (FINAL_FEATURE_B) + leaka...,Fixes the 268-feature final_feature_b set and ...,Guarantees modelling features contain no outco...
5,5. Shared modelling utilities and models,Preprocessing (median impute; scaling fit insi...,Reuses one consistent pipeline and prepares si...
6,6. Metric functions,"Regression, binary, multiclass, calibration, d...",Provides the full metric set the diagnostics a...
7,7. Dependence-robust inference layer,Spearman with Newey-West and moving-block-boot...,Corrects for cross-sectional and serial depend...
8,8. Main experiment runner (fixed train/valid/t...,Loops targets x feature sets x models on the s...,Runs the static-split baseline from one audita...
9,8.1 Simulator-compatible prediction export,Structured JSON/JSONL export with trade.long /...,Produces downstream trade-simulator-compatible...
